# Identification of Bottlenecks at ZRH (LSZH)

## Step 1: Importing Libraries
All required libraries and classes are imported below.

In [ ]:
# Libraries
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
import numpy as np
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# Plotting
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import sample_colorscale
from kaleido import get_chrome_sync

# Traffic
from traffic.core import Traffic
from traffic.data import airports, opensky
from traffic.algorithms.ground.graphs import AirportGraph
from traffic.algorithms.ground.kalman_taxiway import KalmanTaxiway

# Geospatial (preprocessing)
import cartopy.crs as ccrs
import shapely
from shapely.ops import unary_union
from shapely.geometry import (
    Point,
    LineString,
    MultiLineString,
    Polygon,
    MultiPolygon,
    GeometryCollection,
)

## Step 2: Loading Trajectories
In the following cell, the trajectories of aircraft at ZRH are loaded using ADS-B data from the OpenSky Traffic Library. This covers all taxi segments for departing aircraft.

#### <font color='red'> CHANGE THE START AND END TIMES OF THE ANALYSES IN THE CELL BELOW: </font>

In [ ]:
START = ["2025-01-01 00:00"]  # Starting timestamp for the query.
STOP = ["2025-01-02 00:00"]   # Ending timestamp for the query.


bounds = (
    8.343261, # Western side of bounding box.
    47.357783, # Southern side of bounding box.
    8.777561, # Eastern side of bounding box.
    47.568794, # Northern side of bounding box.
)

trajs_filename = ["trajs_raw.parquet"] # List of filenames to save the trajectories.

for i, filename in enumerate(trajs_filename):
    timestamps = pd.date_range(
        start=START[i],
        end=STOP[i],
        freq="24h"
    )

    data = []

    for t1, t2 in tqdm(
        zip(timestamps[:-1], timestamps[1:]),
        total=len(timestamps) - 1,
        desc="Downloading flights"
    ):
        tmp = opensky.history(
            start=str(t1),
            stop=str(t2),
            bounds=bounds,
            #other_params=" and onground=true ", # Query to filter for ground movements.
        )

        if tmp is not None and tmp.data is not None and not tmp.data.empty:
            data.append(tmp.data)

    if not data:
        print("No data found.")
        continue

    trajs = pd.concat(data)

    # Create Traffic object.
    trajs = Traffic(trajs)

In [ ]:
trajs.data.head()

In the following cell, the DataFrame `trajs`, containing all raw trajectories, is saved as a parquet file (`trajs_raw.parquet`).

In [ ]:
output_path = "data/raw/trajs_raw.parquet"

# Save dataframe to parquet.
trajs.to_parquet(output_path, index=False)

print(f"File \"trajs_raw\" saved to: {output_path}")

## Step 3: Filtering Trajectories
In the following cell, the DataFrame `trajs_raw.parquet`, which was created in Step 2 and contains all raw trajectories, is imported.

In [ ]:
trajs = pd.read_parquet("data/raw/trajs_raw.parquet")

# Create Traffic object.
trajs = Traffic(trajs)

In the following cell, the raw trajectories are filtered in a pipeline according to different criteria. The outcome is a set of trajectories containing only the paths of departing aircraft.

In [ ]:
def keep_last_taxi_before_takeoff(flight, airport_code="LSZH", max_taxi_time="45 min"):
    """
    Keeps only the final taxi-out segment before take-off.

    Detects the first take-off segment for the given airport and trims the
    trajectory to the specified time window before lift-off. This helps remove
    earlier ground movements, such as a previous landing, from the same flight
    segment.

    Args:
        flight: The flight trajectory to process.
        airport_code (str, optional): The ICAO code of the departure airport.
            Defaults to "LSZH".
        max_taxi_time (str, optional): The maximum time window to keep before
            take-off. Defaults to "45 min".

    Returns:
        The trimmed flight trajectory, or None if no valid take-off segment is
        found.
    """
    # Detect the first take-off segment.
    takeoff_segment = flight.takeoff(airport_code).next()

    # Discard the trajectory if no take-off segment is found.
    if takeoff_segment is None:
        return None

    # The end of the take-off segment is used as a proxy for lift-off time.
    takeoff_time = takeoff_segment.stop

    # Define how far back in time we want to keep the taxi-out segment.
    taxi_window = pd.Timedelta(max_taxi_time)

    # Keep only the final part of the trajectory before lift-off.
    # This removes earlier movements, such as a previous landing, from the same flight segment.
    trimmed = flight.between(
        takeoff_time - taxi_window,
        takeoff_time,
        strict=False,
    )

    # Discard empty results for safety.
    if trimmed is None or trimmed.data.empty:
        return None

    return trimmed


def trim_parking(flight):
    """
    Removes the initial stationary parking segment from a trajectory.

    Identifies the first period of stable movement using a smoothed ground speed
    signal and trims the trajectory so that it starts when the aircraft begins
    taxiing.

    Args:
        flight: The flight trajectory to process.

    Returns:
        The trimmed flight trajectory starting from the first stable movement,
        or None if no valid movement period is found.
    """
    df = flight.data

    if df.empty:
        return None

    # Use computed groundspeed if available, otherwise use raw groundspeed.
    speed_col = "compute_gs" if "compute_gs" in df.columns else "groundspeed"

    if speed_col not in df.columns:
        return None

    # Fill missing values before smoothing.
    speed = df[speed_col].fillna(0)

    # Smooth the speed with a rolling mean.
    # With 1-second resampled data, window=3 corresponds to 3 seconds.
    speed_smooth = speed.rolling(window=3, min_periods=1).mean()

    # Define the movement threshold in knots.
    moving = speed_smooth > 2

    # Require a stable movement period before considering that the aircraft has actually left the parking position.
    window = 5
    stable_moving = moving.rolling(window=window, min_periods=window).sum() == window

    if not stable_moving.any():
        return None

    # Find the first timestamp with stable movement.
    stable_idx = stable_moving.idxmax()

    # Convert the index position to a row position.
    pos = df.index.get_loc(stable_idx)

    # Step back to the beginning of the stable movement window so that the initial taxi movement is preserved.
    start_pos = max(0, pos - (window - 1))
    start_time = df.iloc[start_pos]["timestamp"]

    # Keep the trajectory from the first stable movement onwards.
    trimmed = flight.after(start_time, strict=False)

    if trimmed is None or trimmed.data.empty:
        return None

    return trimmed


def enough_points_when_taxi(flight, min_taxi_points=30):
    """
    Checks whether a trajectory contains enough taxi points.

    Counts points within a plausible taxi speed range and keeps only trajectories
    with at least the required number of taxi samples.

    Args:
        flight: The flight trajectory to evaluate.
        min_taxi_points (int, optional): The minimum number of taxi points
            required. Defaults to 30.

    Returns:
        The original flight trajectory if enough taxi points are present, or
        None otherwise.
    """
    df = flight.data

    if df.empty:
        return None

    # Use computed groundspeed if available, otherwise use raw groundspeed.
    speed_col = "compute_gs" if "compute_gs" in df.columns else "groundspeed"

    if speed_col not in df.columns:
        return None

    # Define a plausible taxi speed range in knots (1 - 30 knots).
    taxi_mask = (df[speed_col].fillna(0) >= 1) & (df[speed_col].fillna(0) <= 30)

    # Keep only trajectories with enough taxi samples.
    if taxi_mask.sum() < min_taxi_points:
        return None

    return flight


def discard_fast_start_segments(flight, start_points=5, max_start_mean_kt=30):
    """
    Discards trajectories that start at implausibly high speed.

    Computes the mean speed over the first few trajectory points and removes
    segments that are unlikely to represent taxi-out because they already begin
    too fast.

    Args:
        flight: The flight trajectory to evaluate.
        start_points (int, optional): The number of initial points used for the
            mean speed calculation. Defaults to 5.
        max_start_mean_kt (float, optional): The maximum allowed mean start
            speed in knots. Defaults to 30.

    Returns:
        The original flight trajectory if the start speed is plausible, or None
        otherwise.
    """
    df = flight.data

    if df.empty:
        return None

    # Use computed groundspeed if available, otherwise use raw groundspeed.
    speed_col = "compute_gs" if "compute_gs" in df.columns else "groundspeed"

    if speed_col not in df.columns:
        return None

    # If too few points are available, keep the segment as it is.
    if len(df) < start_points:
        return flight

    # Compute the mean speed over the first few points.
    start_mean = df[speed_col].fillna(0).iloc[:start_points].mean()

    # Discard segments that already start too fast to be plausible taxi-out.
    if start_mean > max_start_mean_kt:
        return None

    return flight


def remove_fast_taxi_points(flight):
    """
    Removes individual trajectory points with implausibly high taxi speed.

    Filters out points whose speed exceeds the defined taxi threshold and
    rebuilds the trajectory from the remaining data.

    Args:
        flight: The flight trajectory to process.

    Returns:
        A rebuilt flight trajectory containing only plausible taxi points, or
        None if no valid points remain.
    """
    df = flight.data.copy()

    if df.empty:
        return None

    # Use computed groundspeed if available, otherwise use raw groundspeed.
    speed_col = "compute_gs" if "compute_gs" in df.columns else "groundspeed"

    if speed_col not in df.columns:
        return None

    # Keep only points with plausible taxi speed (<= 40 knots).
    df = df[df[speed_col].fillna(0) <= 40]

    # Return None if no points remain.
    if df.empty:
        return None

    return flight.__class__(df)


# --------------------------- Execute the processing pipeline: ---------------------------

# Step 1:
# Build individual flight segments from the raw OpenSky history data.
trajs_taxi_dep = (
    trajs

    # Remove trajectories with invalid timestamps (quality check).
    # E.g.: Timestamps are out of order, the sequence is incorrect, individual points are implausible.
    .clean_invalid()

    # Assign a flight identifier to each trajectory leg.
    # This helps separating individual flights instead of keeping all points grouped only by aircraft (icao24).
    .assign_id()

    # Add aircraft data to the trajectories.
    .aircraft_data()

    # Keep only trajectories longer than 3 minutes (quality check).
    .longer_than("3 minutes")

    # Execute the segmentation.
    .eval(max_workers=40, desc='Building individual flight segments') # 'max_workers' specifies the number of parallel processes to use for executing the pipeline.
)

# Step 2:
# Extract departure segments.
trajs_taxi_dep = (
    trajs_taxi_dep

    # Select departures from Zurich and exclude local flights.
    #.query("origin == 'LSZH' and destination != origin")

    # Keep trajectories within 5 NM of the middle point of the airport.
    #.distance(airports["LSZH"].point)
    #.query("distance < 5")

    # Keep only flight segments that include a take-off from Zurich.
    .has("takeoff('LSZH')")

    # Ensure trajectories touch runways (quality check).
    .intersects(airports["LSZH"].runways)

    # Standardise the intervals between points.
    .resample("1s")

    # Compute cumulative distance and estimate groundspeed if missing.
    .cumulative_distance(True) # 'True' means: if missing, a ground speed estimate will be added.

    # Keep only the final part of the trajectories before take-off.
    .pipe(keep_last_taxi_before_takeoff)

    # Keep only points within a buffer (in degrees) around the airport layout to remove outliers (quality check).
    .clip(airports["LSZH"].shape.buffer(0.001))

    # Remove points where the aircraft is still parked.
    .pipe(trim_parking)

    # Discard implausible segments that already start too fast (arrivals).
    .pipe(discard_fast_start_segments)

    # Remove trajectories with too few taxi points (quality check).
    .pipe(enough_points_when_taxi)

    # Remove individual points with implausibly high taxi speed (quality check).
    .pipe(remove_fast_taxi_points)

    # Apply the library's standard filter cascade for removing outliers and implausible points (quality check).
    .filter()

    # Execute the full processing pipeline.
    .eval(max_workers=40, desc='Processing flight segments') # 'max_workers' specifies the number of parallel processes to use for executing the pipeline.
)

Comparison on the map between the raw (`trajs`) and the filtered (`trajs_taxi_dep`) trajectories:

Raw (`trajs`):

In [ ]:
MAX_FLIGHTS = 300  # Maximum number of flights to plot for performance reasons.


num_flights = len(trajs)

if num_flights > MAX_FLIGHTS:
    print(f"Too many flights for plotting ({num_flights} > {MAX_FLIGHTS}).")
else:
    fig = go.Figure()

    for flight in trajs:
        fig.add_trace(
            go.Scattermap(
                mode="lines",
                lat=flight.data.latitude,
                lon=flight.data.longitude,
                line=dict(width=2),
                opacity=0.8,
            )
        )

    # Layout
    fig.update_layout(
        width=1000,
        height=1000,
        margin=dict(l=0, r=0, t=0, b=0),
        map=dict(
            style="carto-positron",
            zoom=13,
            center=dict(lat=47.459657, lon=8.550569),
        )
    )

    fig.show()

In [ ]:
MAX_TOTAL_POINTS = 500_000  # Maximum total number of points to plot for performance reasons.
MIN_POINTS_PER_FLIGHT = 2   # Minimum number of points per flight.


# Convert the traffic object to a list of flights.
flights = list(trajs)

valid_flights = []

# Keep only flights with at least the minimum number of valid coordinate points.
for flight in flights:
    df = flight.data[["latitude", "longitude"]].dropna()

    if len(df) >= MIN_POINTS_PER_FLIGHT:
        valid_flights.append(df)

n_flights = len(valid_flights)

if n_flights == 0:
    raise ValueError("No valid flights found.")

# Calculate how many points each flight may use on average.
points_per_flight = max(
    MIN_POINTS_PER_FLIGHT,
    MAX_TOTAL_POINTS // n_flights
)

lat_parts = []
lon_parts = []

for df in valid_flights:
    n = len(df)

    # Reduce long trajectories evenly if they exceed the point budget.
    if n > points_per_flight:
        idx = np.linspace(0, n - 1, points_per_flight, dtype=int)
        df = df.iloc[idx]

    # Convert coordinates to NumPy arrays for better performance.
    lat = df["latitude"].to_numpy(dtype=np.float32)
    lon = df["longitude"].to_numpy(dtype=np.float32)

    lat_parts.append(lat)
    lon_parts.append(lon)

    # Insert NaN separators so Plotly draws separate flight lines.
    lat_parts.append(np.array([np.nan], dtype=np.float32))
    lon_parts.append(np.array([np.nan], dtype=np.float32))

# Merge all flight coordinates into single arrays.
lat_all = np.concatenate(lat_parts)
lon_all = np.concatenate(lon_parts)

fig = go.Figure(
    go.Scattermap(
        mode="lines",
        lat=lat_all,
        lon=lon_all,
        line=dict(width=1),
        opacity=0.8,
        hoverinfo="skip",
        showlegend=False,
    )
)

fig.update_layout(
    width=1000,
    height=1000,
    margin=dict(l=0, r=0, t=0, b=0),
    map=dict(
        style="carto-positron",
        zoom=13,
        center=dict(lat=47.459657, lon=8.550569),
    )
)

fig.show()

Filtered (`trajs_taxi_dep`):

In [ ]:
MAX_FLIGHTS = 300  # Maximum number of flights to plot for performance reasons.


num_flights = len(trajs_taxi_dep)

if num_flights > MAX_FLIGHTS:
    print(f"Too many flights for plotting ({num_flights} > {MAX_FLIGHTS}).")
else:
    fig = go.Figure()

    for flight in trajs_taxi_dep:
        fig.add_trace(
            go.Scattermap(
                mode="lines",
                lat=flight.data.latitude,
                lon=flight.data.longitude,
                line=dict(width=2),
                opacity=0.8,
            )
        )

    # Layout
    fig.update_layout(
        width=1000,
        height=1000,
        margin=dict(l=0, r=0, t=0, b=0),
        map=dict(
            style="carto-positron",
            zoom=13,
            center=dict(lat=47.459657, lon=8.550569),
        )
    )

    fig.show()

In [ ]:
MAX_TOTAL_POINTS = 500_000  # Maximum total number of points to plot for performance reasons.
MIN_POINTS_PER_FLIGHT = 2   # Minimum number of points per flight.


# Convert the traffic object to a list of flights.
flights = list(trajs_taxi_dep)

valid_flights = []

# Keep only flights with at least the minimum number of valid coordinate points.
for flight in flights:
    df = flight.data[["latitude", "longitude"]].dropna()

    if len(df) >= MIN_POINTS_PER_FLIGHT:
        valid_flights.append(df)

n_flights = len(valid_flights)

if n_flights == 0:
    raise ValueError("No valid flights found.")

# Calculate how many points each flight may use on average.
points_per_flight = max(
    MIN_POINTS_PER_FLIGHT,
    MAX_TOTAL_POINTS // n_flights
)

lat_parts = []
lon_parts = []

for df in valid_flights:
    n = len(df)

    # Reduce long trajectories evenly if they exceed the point budget.
    if n > points_per_flight:
        idx = np.linspace(0, n - 1, points_per_flight, dtype=int)
        df = df.iloc[idx]

    # Convert coordinates to NumPy arrays for better performance.
    lat = df["latitude"].to_numpy(dtype=np.float32)
    lon = df["longitude"].to_numpy(dtype=np.float32)

    lat_parts.append(lat)
    lon_parts.append(lon)

    # Insert NaN separators so Plotly draws separate flight lines.
    lat_parts.append(np.array([np.nan], dtype=np.float32))
    lon_parts.append(np.array([np.nan], dtype=np.float32))

# Merge all flight coordinates into single arrays.
lat_all = np.concatenate(lat_parts)
lon_all = np.concatenate(lon_parts)

fig = go.Figure(
    go.Scattermap(
        mode="lines",
        lat=lat_all,
        lon=lon_all,
        line=dict(width=1),
        opacity=0.8,
        hoverinfo="skip",
        showlegend=False,
    )
)

fig.update_layout(
    width=1000,
    height=1000,
    margin=dict(l=0, r=0, t=0, b=0),
    map=dict(
        style="carto-positron",
        zoom=13,
        center=dict(lat=47.459657, lon=8.550569),
    )
)

fig.show()

In [ ]:
trajs_taxi_dep

In [ ]:
trajs_taxi_dep.data.head()

## Step 4: Preprocessing Trajectories
In the following cells, the trajectories are preprocessed in a pipeline. The outcome is clean taxi-out trajectories (from Off-Block to Runway Line-Up) for all aircraft, which provides usable data for identifying bottlenecks in the taxi-out system.

#### <font color='red'> CHANGE THE GLOBAL PROCESSING AND SMOOTHING PARAMETERS IN THE CELL BELOW: </font>

In [ ]:
MAX_WORKERS = 40            # Number of parallel worker processes used for flight processing.

MIN_CONSECUTIVE_POINTS = 5  # Minimum number of consecutive trajectory points that must lie inside a runway threshold polygon to count as a valid hit.

POS_EPS_M = 0.5             # [m] Minimum movement distance below which the aircraft is treated as stationary.
HEADING_WINDOW = 2          # Number of points before and after each sample used to compute speed and track.

POS_EPS_M_KALMAN = 0.5      # [m] Minimum movement distance below which Kalman-smoothed motion is treated as stationary.
HEADING_WINDOW_KALMAN = 2   # Number of points before and after each sample used for Kalman-based speed and track computation.
ROLLING_WINDOW_KALMAN = 5   # Window size (should be uneven!) used to smooth Kalman-based speed and track values.

In the following cell:
- The airport code and airport reference coordinates are set up.
- The airport graph is constructed in a local projected coordinate system.
- Available graph repair methods are applied to clean the airport graph.
- Threshold polygons are defined for each runway.

In [ ]:
airport_code = "LSZH"


def airport_lon_lat(airport):
    """
    Extracts the longitude and latitude of an airport.

    Tries common airport coordinate attributes first and falls back to the
    airport point geometry if necessary.

    Args:
        airport: The airport object from which to extract coordinates.

    Returns:
        tuple[float, float]: The airport longitude and latitude.

    Raises:
        RuntimeError: If the airport coordinates cannot be determined.
    """
    # Try the most common airport longitude/latitude attribute names first.
    for lon_attr, lat_attr in (("longitude", "latitude"), ("lon", "lat")):
        lon = getattr(airport, lon_attr, None)
        lat = getattr(airport, lat_attr, None)
        if lon is not None and lat is not None:
            return float(lon), float(lat)

    # Fall back to the airport point geometry if direct attributes are unavailable.
    point = getattr(airport, "point", None)
    if point is not None:
        for lon_attr, lat_attr in (("longitude", "latitude"), ("lon", "lat"), ("x", "y")):
            lon = getattr(point, lon_attr, None)
            lat = getattr(point, lat_attr, None)
            if lon is not None and lat is not None:
                return float(lon), float(lat)

    raise RuntimeError("Could not determine airport coordinates.")


# Build the airport graph in a local projected coordinate system.
lon, lat = airport_lon_lat(airports["LSZH"])

graph = AirportGraph.from_airport(
    airports["LSZH"],
    ccrs.AzimuthalEquidistant(
        central_longitude=lon,
        central_latitude=lat,
    ),
)

# Clean the graph if the corresponding repair methods are available.
for method_name in ("merge_duplicate_nodes", "fix_airport_graph"):
    method = getattr(graph, method_name, None)
    if callable(method):
        try:
            updated_graph = method()
            if updated_graph is not None:
                graph = updated_graph
        except Exception as e:
            print(f"Warning: graph method '{method_name}' failed: {e}")

# Define threshold polygons for each runway.
# Coordinates must be given as (longitude, latitude).
threshold_polygons = {
    "28": Polygon([
        (8.570553, 47.456850),
        (8.570485, 47.456342),
        (8.563255, 47.456850),
        (8.563296, 47.457367),
    ]),
    "32": Polygon([
        (8.555662, 47.468149),
        (8.564743, 47.461497),
        (8.564220, 47.461026),
        (8.555302, 47.467577),
    ]),
    "16": Polygon([
        (8.535351, 47.475783),
        (8.536171, 47.475995),
        (8.544613, 47.463705),
        (8.543875, 47.463493),
    ]),
    "10": Polygon([
        (8.537393, 47.459241),
        (8.546667, 47.458548),
        (8.546613, 47.457994),
        (8.537297, 47.458650),
    ]),
    "34": Polygon([
        (8.553569, 47.450691),
        (8.557139, 47.445614),
        (8.556352, 47.445410),
        (8.552860, 47.450487),
    ]),
}

In the following cell:
- Helper functions for rebuilding traffic-like objects from DataFrames are defined.
- Groundspeed and track are recomputed from projected x/y positions.
- A geometric mask is used to determine whether points lie inside a polygon or on its boundary.
- The first runway threshold hit is detected based on consecutive trajectory points within a threshold polygon.

In [ ]:
def rebuild_like(template_or_cls, df):
    """
    Rebuilds a traffic-like object from a DataFrame.

    Tries common constructor patterns to recreate an object of the same type as
    the provided template or class. Falls back to returning the DataFrame if no
    suitable constructor is available.

    Args:
        template_or_cls: A template object or class used to determine the target
            type.
        df (pandas.DataFrame): The DataFrame from which to rebuild the object.

    Returns:
        The rebuilt object, or the original DataFrame if reconstruction is not
        possible.
    """
    cls = template_or_cls if isinstance(template_or_cls, type) else template_or_cls.__class__

    try:
        return cls(df)
    except Exception:
        pass

    from_dataframe = getattr(cls, "from_dataframe", None)
    if callable(from_dataframe):
        return from_dataframe(df)

    # Fall back to the dataframe itself if no suitable constructor exists.
    return df


def recompute_speed_track_from_xy(
    df,
    x_col,
    y_col,
    groundspeed_col,
    track_col,
    pos_eps_m=None,
    heading_window=None,
    inplace=False,
):
    """
    Recomputes ground speed and track from projected x/y positions.

    Uses centred finite differences over projected coordinates to estimate local
    movement direction and speed, then stores the results in the specified
    output columns.

    Args:
        df (pandas.DataFrame): The input trajectory DataFrame.
        x_col (str): The name of the x-coordinate column.
        y_col (str): The name of the y-coordinate column.
        groundspeed_col (str): The name of the output ground speed column.
        track_col (str): The name of the output track column.
        pos_eps_m (float, optional): The minimum movement distance in metres
            below which the aircraft is treated as stationary. Defaults to None.
        heading_window (int, optional): The number of points before and after
            each sample used for the centred difference. Defaults to None.
        inplace (bool, optional): Whether to modify the input DataFrame in
            place. Defaults to False.

    Returns:
        pandas.DataFrame: The DataFrame with recomputed ground speed and track
        columns.
    """
    if pos_eps_m is None:
        pos_eps_m = POS_EPS_M
    if heading_window is None:
        heading_window = HEADING_WINDOW

    out = df if inplace else df.copy()

    if out.empty:
        out[groundspeed_col] = pd.Series(index=out.index, dtype=float)
        out[track_col] = pd.Series(index=out.index, dtype=float)
        return out

    ts = pd.to_datetime(out["timestamp"], errors="coerce").to_numpy()
    x = out[x_col].to_numpy(dtype=float, copy=False)
    y = out[y_col].to_numpy(dtype=float, copy=False)

    n = len(out)
    groundspeed_vals = np.full(n, np.nan, dtype=float)
    track_vals = np.full(n, np.nan, dtype=float)

    # With a centred window of size heading_window, each point i is compared against points i-heading_window and i+heading_window.
    # Near the edges, the nearest valid window is used instead.
    idx = np.arange(n)
    j = np.maximum(0, idx - heading_window)
    k = np.minimum(n - 1, idx + heading_window)

    valid_time = (~pd.isna(ts[j])) & (~pd.isna(ts[k])) & (j != k)
    if valid_time.any():
        dt = np.full(n, np.nan, dtype=float)
        dt[valid_time] = (ts[k[valid_time]] - ts[j[valid_time]]) / np.timedelta64(1, "s")

        dx = x[k] - x[j]
        dy = y[k] - y[j]
        dist = np.hypot(dx, dy)

        moving_mask = valid_time & (dt > 0) & (dist >= pos_eps_m)
        stationary_mask = valid_time & (dt > 0) & (dist < pos_eps_m)

        groundspeed_vals[stationary_mask] = 0.0
        track_vals[stationary_mask] = np.nan

        # Convert m/s to knots by multiplying with 1.94384449.
        groundspeed_vals[moving_mask] = (dist[moving_mask] / dt[moving_mask]) * 1.94384449
        track_vals[moving_mask] = (
            np.degrees(np.arctan2(dx[moving_mask], dy[moving_mask])) + 360.0
        ) % 360.0

    gs_series = pd.Series(groundspeed_vals, index=out.index)
    track_series = pd.Series(track_vals, index=out.index)

    # Keep missing groundspeed values as NaN (unknown). Set track to NaN for stationary points only.
    out[groundspeed_col] = gs_series
    out[track_col] = track_series
    out.loc[out[groundspeed_col] <= 0.0, track_col] = np.nan

    return out


def _inside_or_boundary_mask(poly, lon_vals, lat_vals):
    """
    Creates a mask indicating whether points lie inside or on a polygon boundary.

    Applies a fast bounding-box prefilter and then uses Shapely vectorised
    geometry checks for the remaining candidate points.

    Args:
        poly: The polygon used for the spatial test.
        lon_vals: The longitude values of the points.
        lat_vals: The latitude values of the points.

    Returns:
        numpy.ndarray: A boolean array indicating whether each point is inside
        or on the boundary of the polygon.
    """
    # Performance strategy:
    # 1. Apply a computationally inexpensive bounding-box prefilter in NumPy.
    # 2. Only test the remaining candidate points with Shapely vectorised predicates.
    minx, miny, maxx, maxy = poly.bounds
    bbox_mask = (
        (lon_vals >= minx) & (lon_vals <= maxx) &
        (lat_vals >= miny) & (lat_vals <= maxy)
    )

    if not bbox_mask.any():
        return np.zeros(len(lon_vals), dtype=bool)

    result = np.zeros(len(lon_vals), dtype=bool)
    cand_idx = np.flatnonzero(bbox_mask)
    pts = shapely.points(lon_vals[cand_idx], lat_vals[cand_idx])
    result[cand_idx] = np.asarray(shapely.covers(poly, pts), dtype=bool)

    return result


def first_runway_threshold_hit(df, threshold_polygons, min_consecutive_points=None):
    """
    Finds the first runway threshold reached by a trajectory.

    Tests each runway threshold polygon and identifies the earliest timestamp at
    which a sufficient number of consecutive trajectory points fall within a
    threshold area.

    Args:
        df (pandas.DataFrame): The trajectory data.
        threshold_polygons (dict): A dictionary mapping runway identifiers to
            threshold polygons.
        min_consecutive_points (int, optional): The minimum number of
            consecutive points required for a valid threshold hit. Defaults to
            None.

    Returns:
        tuple: A pair containing the first hit timestamp and the corresponding
        runway identifier, or (None, None) if no threshold hit is found.
    """
    if min_consecutive_points is None:
        min_consecutive_points = MIN_CONSECUTIVE_POINTS

    if df.empty:
        return None, None

    required_cols = {"longitude", "latitude", "timestamp"}
    if not required_cols.issubset(df.columns):
        return None, None

    work = df.loc[:, ["timestamp", "longitude", "latitude"]].dropna(
        subset=["timestamp", "longitude", "latitude"]
    )
    if work.empty:
        return None, None

    ts = pd.to_datetime(work["timestamp"], errors="coerce")
    valid_ts = ts.notna().to_numpy()
    if not valid_ts.any():
        return None, None

    ts_vals = ts[valid_ts].to_numpy()
    lon_vals = work.loc[valid_ts, "longitude"].to_numpy(dtype=float, copy=False)
    lat_vals = work.loc[valid_ts, "latitude"].to_numpy(dtype=float, copy=False)

    first_hits = []

    kernel = np.ones(min_consecutive_points, dtype=int)

    for rwy, poly in threshold_polygons.items():
        inside = _inside_or_boundary_mask(poly, lon_vals, lat_vals).astype(int)

        if len(inside) < min_consecutive_points:
            continue

        consecutive_hits = np.convolve(inside, kernel, mode="valid") >= min_consecutive_points
        if consecutive_hits.any():
            # The first valid window starts at index start_idx.
            # The first timestamp matching the "MIN_CONSECUTIVE_POINTS reached" logic is the end of that window.
            start_idx = int(np.argmax(consecutive_hits))
            first_idx = start_idx + min_consecutive_points - 1
            first_hits.append((ts_vals[first_idx], rwy))

    if not first_hits:
        return None, None

    cut_time, cut_runway = min(first_hits, key=lambda x: x[0])

    return cut_time, cut_runway

In the following cell:
- A processing pipeline for a single flight trajectory is defined.
- Each trajectory is checked for a runway threshold crossing and cut accordingly (`cut_runway`).
- The trajectory is projected into x/y coordinates (`x` and `y`).
- Groundspeed and track are recomputed from the projected positions (`groundspeed_xy` and `track_xy`).
- The Kalman filter is applied in x/y space.
- The filtered trajectory is transformed back to latitude/longitude (`latitude_kalman` and `longitude_kalman`).
- Kalman-based groundspeed and track are recomputed and smoothed (`groundspeed_kalman` and `track_kalman`).

In [ ]:
def process_one_flight(
    df,
    threshold_polygons,
    airport_code,
    projection,
    flight_cls,
    pos_eps_m_kalman=None,
    heading_window_kalman=None,
    rolling_window=None,
    min_consecutive_points=None,
):
    """
    Processes a single flight trajectory through the threshold and Kalman pipeline.

    Detects the first runway threshold hit, cuts the trajectory before the
    threshold, projects it into x/y space, recomputes motion variables, applies
    Kalman smoothing, and converts the result back to latitude/longitude.

    Args:
        df (pandas.DataFrame): The trajectory data for one flight.
        threshold_polygons (dict): A dictionary mapping runway identifiers to
            threshold polygons.
        airport_code (str): The ICAO code of the airport.
        projection: The projection used for x/y coordinate conversion.
        flight_cls: The flight class used to rebuild traffic-like objects.
        pos_eps_m_kalman (float, optional): The minimum movement distance in
            metres for Kalman-based motion estimation. Defaults to None.
        heading_window_kalman (int, optional): The centred difference window
            size for Kalman-based motion estimation. Defaults to None.
        rolling_window (int, optional): The rolling window used to smooth
            Kalman-based speed and track values. Defaults to None.
        min_consecutive_points (int, optional): The minimum number of
            consecutive threshold points required. Defaults to None.

    Returns:
        dict | None: A dictionary containing the processed DataFrame and runway
        identifier, or None if processing fails.
    """
    try:
        if pos_eps_m_kalman is None:
            pos_eps_m_kalman = POS_EPS_M_KALMAN
        if heading_window_kalman is None:
            heading_window_kalman = HEADING_WINDOW_KALMAN
        if rolling_window is None:
            rolling_window = ROLLING_WINDOW_KALMAN
        if min_consecutive_points is None:
            min_consecutive_points = MIN_CONSECUTIVE_POINTS

        if df is None or df.empty:
            return None

        if not {"latitude", "longitude", "timestamp"}.issubset(df.columns):
            return None

        # Detect first runway threshold hit.
        cut_time, cut_runway = first_runway_threshold_hit(
            df,
            threshold_polygons=threshold_polygons,
            min_consecutive_points=min_consecutive_points,
        )

        if cut_time is None:
            return None

        # Cut trajectory using DataFrame filtering.
        cut_df = df[df["timestamp"] < cut_time].copy()
        if cut_df.empty:
            return None

        # Store runway identifier ("cut_runway").
        cut_df["cut_runway"] = cut_runway

        # Rebuild a minimal flight-like object only where required for projection.
        flight_xy = rebuild_like(flight_cls, cut_df).compute_xy(projection)

        # Keep the ordering stable for all downstream finite-difference calculations.
        df_xy = flight_xy.data
        if not df_xy["timestamp"].is_monotonic_increasing:
            df_xy = df_xy.sort_values("timestamp").reset_index(drop=True)
        else:
            df_xy = df_xy.reset_index(drop=True)

        # Recompute speed and track based on projected coordinates.
        recompute_speed_track_from_xy(
            df_xy,
            x_col="x",
            y_col="y",
            groundspeed_col="groundspeed_xy",
            track_col="track_xy",
            inplace=True,
        )

        # Prepare minimal input for Kalman filter.
        df_kalman_input = pd.DataFrame(
            {
                "timestamp": df_xy["timestamp"].to_numpy(copy=False),
                "x": df_xy["x"].to_numpy(copy=False),
                "y": df_xy["y"].to_numpy(copy=False),
                "groundspeed": df_xy["groundspeed_xy"].to_numpy(copy=False),
                "track": df_xy["track_xy"].to_numpy(copy=False),
            }
        )

        # Apply Kalman filter in x/y space.
        kalman = KalmanTaxiway(airport_code, projection)
        df_kalman = kalman.apply(df_kalman_input)

        if df_kalman is None or df_kalman.empty:
            return None

        # Prepare Kalman output for inverse projection.
        kalman_xy_df = pd.DataFrame(
            {
                "timestamp": df_xy["timestamp"].to_numpy(copy=False),
                "x": df_kalman["x"].to_numpy(copy=False),
                "y": df_kalman["y"].to_numpy(copy=False),
            }
        )

        # Convert back to latitude/longitude.
        kalman_latlon = (
            rebuild_like(flight_cls, kalman_xy_df)
            .compute_latlon_from_xy(projection)
            .data.loc[:, ["latitude", "longitude"]]
            .reset_index(drop=True)
        )

        # Store Kalman-smoothed positions.
        df_xy["latitude_kalman"] = kalman_latlon["latitude"].to_numpy(copy=False)
        df_xy["longitude_kalman"] = kalman_latlon["longitude"].to_numpy(copy=False)

        # Compute Kalman-based speed and track from the Kalman-filtered x/y coordinates.
        # This uses a centred finite-difference window in order to obtain a more stable local direction.
        df_kalman_xy = recompute_speed_track_from_xy(
            kalman_xy_df,
            x_col="x",
            y_col="y",
            groundspeed_col="groundspeed_kalman",
            track_col="track_kalman",
            pos_eps_m=pos_eps_m_kalman,
            heading_window=heading_window_kalman,
            inplace=False,
        )

        groundspeed_k = (
            pd.Series(df_kalman_xy["groundspeed_kalman"])
            .rolling(window=rolling_window, center=True, min_periods=1)
            .mean()
            .to_numpy()
        )

        track_k = df_kalman_xy["track_kalman"].to_numpy(copy=False)
        track_rad = np.deg2rad(track_k)
        track_sin = np.sin(track_rad)
        track_cos = np.cos(track_rad)

        track_sin_smooth = (
            pd.Series(track_sin)
            .rolling(window=rolling_window, center=True, min_periods=1)
            .mean()
            .to_numpy()
        )
        track_cos_smooth = (
            pd.Series(track_cos)
            .rolling(window=rolling_window, center=True, min_periods=1)
            .mean()
            .to_numpy()
        )

        track_k = (np.degrees(np.arctan2(track_sin_smooth, track_cos_smooth)) + 360.0) % 360.0

        # Forward/backward filling improves continuity near the trajectory edges.
        # Track remains undefined for stationary points and is therefore cleared again below.
        groundspeed_series = pd.Series(groundspeed_k, index=df_xy.index)
        track_series = pd.Series(track_k, index=df_xy.index)

        df_xy["groundspeed_kalman"] = groundspeed_series.ffill().bfill()
        df_xy["track_kalman"] = track_series.ffill().bfill()
        df_xy.loc[df_xy["groundspeed_kalman"] <= 0.0, "track_kalman"] = np.nan

        return {
            "df": df_xy,
            "cut_runway": cut_runway,
        }

    except Exception:
        # Silent fail to avoid I/O overhead in parallel execution.
        return None

In the following cell:
- Flight trajectories are extracted as DataFrames for lightweight processing.
- The flight class is identified once for later reconstruction of traffic-like objects.
- All flights are processed in parallel using a bounded number of worker processes (`MAX_WORKERS`).
- The processed trajectories are collected and summarised by runway threshold cut (`cut_runway`).
- All processed flight trajectories are merged into a final traffic object (`trajs_taxi_out_processed`).

In [ ]:
processed_dfs = []
cut_runway_stats = {rwy: 0 for rwy in threshold_polygons.keys()}

# Store the flight class explicitly once.
flight_cls = next(
    (f.__class__ for f in trajs_taxi_dep if f is not None and hasattr(f, "data")),
    None,
)
if flight_cls is None:
    raise ValueError("No valid flight object found in trajs_taxi_dep.")

# Extract only DataFrames (lighter than full flight objects).
flight_dfs = [
    f.data for f in trajs_taxi_dep
    if f is not None and hasattr(f, "data")
]

# Keep only a bounded number of tasks in flight at the same time.
# This reduces peak RAM usage when many trajectories are processed.
MAX_IN_FLIGHT = max(MAX_WORKERS * 2, 1)

# Parallel execution across flights.
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    flight_iter = iter(flight_dfs)
    futures = {}

    # Submit an initial bounded batch.
    for _ in range(min(MAX_IN_FLIGHT, len(flight_dfs))):
        df = next(flight_iter, None)
        if df is None:
            break

        fut = executor.submit(
            process_one_flight,
            df,
            threshold_polygons,
            airport_code,
            graph.projection,
            flight_cls,
        )
        futures[fut] = None

    # Collect results as they complete and keep refilling the queue.
    while futures:
        done_future = next(as_completed(futures))
        futures.pop(done_future, None)

        try:
            res = done_future.result()
        except Exception as e:
            print(f"Worker failed: {e}")
            res = None
        if res is not None:
            processed_dfs.append(res["df"])
            cut_runway_stats[res["cut_runway"]] += 1

        next_df = next(flight_iter, None)
        if next_df is not None:
            fut = executor.submit(
                process_one_flight,
                next_df,
                threshold_polygons,
                airport_code,
                graph.projection,
                flight_cls,
            )
            futures[fut] = None

# Combine all processed trajectories.
if processed_dfs:
    combined_df = pd.concat(processed_dfs, ignore_index=True, sort=False)
    trajs_taxi_out_processed = rebuild_like(trajs_taxi_dep, combined_df)
else:
    trajs_taxi_out_processed = None

In the following cell:
- Airport geometries are extracted for the relevant layout layers.
- A short processing summary is printed, including threshold and Kalman success rates.
- The airport layout is plotted as the base map.
- The Kalman-snapped taxi-out trajectories are visualised.
- The end point of each trajectory is marked.

In [ ]:
def is_geom(x):
    """
    Checks whether an object is a supported Shapely geometry.

    Args:
        x: The object to test.

    Returns:
        bool: True if the object is a supported Shapely geometry, otherwise
        False.
    """
    return isinstance(
        x,
        (
            Point,
            LineString,
            MultiLineString,
            Polygon,
            MultiPolygon,
            GeometryCollection,
        ),
    )


def extract_geoms(obj):
    """
    Extracts valid Shapely geometries from an object.

    Supports direct geometry objects, objects with shape or geometry
    attributes, and DataFrames containing a geometry column.

    Args:
        obj: The object from which to extract geometries.

    Returns:
        list: A list of valid, non-empty Shapely geometries.
    """
    if obj is None:
        return []

    if is_geom(obj):
        return [obj]

    shape = getattr(obj, "shape", None)
    if is_geom(shape):
        return [shape]

    geometry = getattr(obj, "geometry", None)
    if geometry is not None:
        if is_geom(geometry):
            return [geometry]
        try:
            return [g for g in geometry if g is not None and is_geom(g) and not g.is_empty]
        except Exception:
            pass

    data = getattr(obj, "data", None)
    if isinstance(data, pd.DataFrame) and "geometry" in data.columns:
        return [
            g for g in data["geometry"]
            if g is not None and is_geom(g) and not g.is_empty
        ]

    return []


def airport_geom(airport, kind):
    """
    Collects and merges airport geometries for a given visual layer.

    Looks up the relevant airport attributes for the requested layer type and
    combines the extracted geometries into a single geometry where possible.

    Args:
        airport: The airport object containing geometry information.
        kind (str): The type of visual layer to extract, such as "taxiways",
            "apron", "movement_area", "runways", or "shape".

    Returns:
        The merged geometry for the requested layer, or None if no geometry is
        available.
    """
    attr_map = {
        "taxiways": ["taxiways", "taxiway"],
        "apron": ["apron"],
        "movement_area": ["taxiways", "taxiway", "apron", "layout"],
        "runways": ["runways"],
        "shape": ["shape"],
    }

    geoms = []
    for attr in attr_map[kind]:
        geoms.extend(extract_geoms(getattr(airport, attr, None)))

    if not geoms:
        return None

    try:
        return unary_union(geoms)
    except Exception:
        return geoms[0]


def add_geom(
    fig,
    geom,
    name,
    line_width=1,
    line_colour="grey",
    fill_colour=None,
    opacity=1.0,
    showlegend=False,
):
    """
    Adds a Shapely geometry to a Plotly figure.

    Supports line and polygon geometries, including multi-part and collection
    types, and renders them with the specified styling options.

    Args:
        fig: The Plotly figure to which the geometry is added.
        geom: The Shapely geometry to plot.
        name (str): The legend name for the geometry.
        line_width (int | float, optional): The line width used for plotting.
            Defaults to 1.
        line_colour (str, optional): The line colour used for plotting.
            Defaults to "grey".
        fill_colour (str, optional): The fill colour for polygon geometries.
            Defaults to None.
        opacity (float, optional): The opacity used for plotting. Defaults to
            1.0.
        showlegend (bool, optional): Whether to show the geometry in the
            legend. Defaults to False.

    Returns:
        None
    """
    if geom is None or not is_geom(geom) or geom.is_empty:
        return

    def add_line(x, y, show):
        fig.add_trace(
            go.Scatter(
                mode="lines",
                x=np.asarray(x, dtype=np.float32),
                y=np.asarray(y, dtype=np.float32),
                name=name,
                showlegend=show,
                line=dict(width=line_width, color=line_colour),
                opacity=opacity,
                hoverinfo="skip",
            )
        )

    def add_poly(x, y, show):
        fig.add_trace(
            go.Scatter(
                mode="lines",
                x=np.asarray(x, dtype=np.float32),
                y=np.asarray(y, dtype=np.float32),
                name=name,
                showlegend=show,
                line=dict(width=line_width, color=line_colour),
                fill="toself",
                fillcolor=fill_colour if fill_colour is not None else line_colour,
                opacity=opacity,
                hoverinfo="skip",
            )
        )

    first = True

    if isinstance(geom, LineString):
        x, y = geom.xy
        add_line(x, y, showlegend)
        return

    if isinstance(geom, MultiLineString):
        for part in geom.geoms:
            x, y = part.xy
            add_line(x, y, showlegend if first else False)
            first = False
        return

    if isinstance(geom, Polygon):
        x, y = geom.exterior.xy
        add_poly(x, y, showlegend)
        return

    if isinstance(geom, MultiPolygon):
        for part in geom.geoms:
            x, y = part.exterior.xy
            add_poly(x, y, showlegend if first else False)
            first = False
        return

    if isinstance(geom, GeometryCollection):
        for part in geom.geoms:
            add_geom(
                fig,
                part,
                name,
                line_width=line_width,
                line_colour=line_colour,
                fill_colour=fill_colour,
                opacity=opacity,
                showlegend=showlegend if first else False,
            )
            first = False


processed_df_for_output = None
if trajs_taxi_out_processed is not None:
    processed_df_for_output = (
        trajs_taxi_out_processed.data
        if hasattr(trajs_taxi_out_processed, "data")
        else trajs_taxi_out_processed
    )

# Print the processing summary.
original_flights = len(trajs_taxi_dep)
threshold_flights = sum(cut_runway_stats.values())

if isinstance(processed_df_for_output, pd.DataFrame) and not processed_df_for_output.empty:
    processed_flights = processed_df_for_output["flight_id"].dropna().nunique()
else:
    processed_flights = 0

threshold_rate = (threshold_flights / original_flights) * 100 if original_flights > 0 else 0.0
kalman_rate_after_cut = (processed_flights / threshold_flights) * 100 if threshold_flights > 0 else 0.0
overall_rate = (processed_flights / original_flights) * 100 if original_flights > 0 else 0.0

print("--------------------------------------------------")
print("INPUT")
print(f"Original flights: {original_flights}")
print("--------------------------------------------------")
print("THRESHOLD CUT")
print(f"Flights reaching threshold: {threshold_flights}")
print(f"Threshold reach rate: {threshold_rate:.1f}%")
print(f"Flights cut at RWY 28: {cut_runway_stats['28']}")
print(f"Flights cut at RWY 32: {cut_runway_stats['32']}")
print(f"Flights cut at RWY 16: {cut_runway_stats['16']}")
print(f"Flights cut at RWY 10: {cut_runway_stats['10']}")
print(f"Flights cut at RWY 34: {cut_runway_stats['34']}")
print("--------------------------------------------------")
print("KALMAN PROCESSING")
print(f"Kalman processed flights after threshold cut: {processed_flights}")
print(f"Kalman success after threshold cut: {kalman_rate_after_cut:.1f}%")
print("--------------------------------------------------")
print("OVERALL")
print(f"Overall processing success rate: {overall_rate:.1f}%")
print("--------------------------------------------------")


# Create the base figure.
fig = go.Figure()

# Draw airport layout layers first.
layers = [
    ("shape", "Buildings", 1.0, "rgba(0, 0, 0, 1.00)", "rgba(0, 0, 0, 1.00)", 1.00),
    ("movement_area", "Apron & taxiways", 1.0, "rgba(180, 180, 180, 0.90)", "rgba(220, 220, 220, 0.60)", 0.95),
    ("runways", "Runways", 3, "rgba(20, 20, 20, 0.95)", "rgba(40, 40, 40, 0.20)", 0.95),
]

for kind, name, line_width, line_colour, fill_colour, opacity in layers:
    geom = airport_geom(airports["LSZH"], kind)
    if geom is not None:
        add_geom(
            fig,
            geom,
            name=name,
            line_width=line_width,
            line_colour=line_colour,
            fill_colour=fill_colour,
            opacity=opacity,
            showlegend=True,
        )

# Collect all taxi-out lines in one trace and all end points in one marker trace.
line_x_parts = []
line_y_parts = []
end_x = []
end_y = []

if isinstance(processed_df_for_output, pd.DataFrame) and not processed_df_for_output.empty:
    for _, flight_df in processed_df_for_output.groupby("flight_id", sort=False):
        # Use the dedicated Kalman longitude/latitude columns for plotting.
        df_plot = flight_df.loc[:, ["longitude_kalman", "latitude_kalman"]].dropna()
        if len(df_plot) < 2:
            continue

        lon = df_plot["longitude_kalman"].to_numpy(dtype=np.float32)
        lat = df_plot["latitude_kalman"].to_numpy(dtype=np.float32)

        # Append each trajectory and separate consecutive flights with NaN values.
        line_x_parts.append(lon)
        line_y_parts.append(lat)
        line_x_parts.append(np.array([np.nan], dtype=np.float32))
        line_y_parts.append(np.array([np.nan], dtype=np.float32))

        # Store the final point of each taxi-out trajectory.
        end_x.append(lon[-1])
        end_y.append(lat[-1])

# Plot all snapped taxi-out trajectories in a single WebGL trace.
if line_x_parts:
    fig.add_trace(
        go.Scattergl(
            mode="lines",
            x=np.concatenate(line_x_parts),
            y=np.concatenate(line_y_parts),
            name="Kalman-snapped taxi-out",
            showlegend=True,
            line=dict(width=1, color="rgba(0, 90, 255, 1.00)"),
            opacity=1.00,
            hoverinfo="skip",
        )
    )

# Plot all taxi-out end points in a single WebGL marker trace.
if end_x:
    fig.add_trace(
        go.Scattergl(
            mode="markers",
            x=np.asarray(end_x, dtype=np.float32),
            y=np.asarray(end_y, dtype=np.float32),
            name="Taxi-out end",
            showlegend=True,
            marker=dict(size=10, symbol="x", color="rgba(200, 30, 30, 0.98)"),
            hovertemplate="Longitude: %{x:.6f}<br>Latitude: %{y:.6f}<extra>Taxi-out end</extra>",
        )
    )

fig.update_layout(
    title="Kalman-Snapped Taxi-Out Trajectories",
    width=1000,
    height=1000,
    margin=dict(l=20, r=20, t=50, b=20),
    legend=dict(orientation="h"),
    template="plotly_white",
)

# Keep the axes at the same scale so that the airport geometry is not distorted.
fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

In [ ]:
trajs_taxi_out_processed.data.head()

In the following cell, the DataFrame `trajs_taxi_out_processed`, containing all processed taxi-out trajectories, is saved as a parquet file (`trajs_processed.parquet`).

In [ ]:
output_path = "data/processed/trajs_processed.parquet"

# Save dataframe to parquet.
trajs_taxi_out_processed.to_parquet(output_path, index=False)

print(f"File \"trajs_processed\" saved to: {output_path}")

## Step 5: Heat Maps
In the following cells, maps are generated to visually identify bottlenecks at ZRH.

### Bottleneck Heatmap
<font color='red'> The desired take-off runway can be entered in the first line of code. </font>

In [ ]:
# ========================= Adjust the parameters here inside =========================

RUNWAY = "28"           # Runway to analyse (28, 32, 16, 10 or 34).

SPEED_THRESHOLD = 1.0   # [kt] Speed threshold for bottleneck detection.
MIN_SAMPLES = 1*60      # Minimum number of samples (data points) required per cell.
CELL_SIZE = 0.00050     # [deg] Size of the heatmap cells in longitude/latitude degrees.
MARKER_SIZE = 8         # Size of the heatmap markers.

# ========================= Adjust the parameters here inside =========================


df = trajs_taxi_out_processed.data.copy()
df = df[df["cut_runway"] == RUNWAY]
df = df[df["groundspeed_kalman"] < SPEED_THRESHOLD]

# Keep only rows with valid Kalman coordinates and speed values.
df = df.dropna(
    subset=[
        "longitude_kalman",
        "latitude_kalman",
        "groundspeed_kalman",
    ]
)

if df.empty:
    print("No bottleneck data found.")
else:
    # Build raster cells in longitude/latitude space.
    df["cell_lon"] = np.floor(df["longitude_kalman"] / CELL_SIZE)
    df["cell_lat"] = np.floor(df["latitude_kalman"] / CELL_SIZE)

    # Aggregate total holding time per cell (number of points corresponds to time spent).
    # Divide the time by the number of unique flights affected in that cell.
    agg = (
        df.groupby(["cell_lon", "cell_lat"])
        .agg(
            duration_s=("timestamp", "size"),
            n_flights=("flight_id", "nunique"),
        )
        .reset_index()
    )

    # Remove cells with too few samples.
    agg = agg[agg["duration_s"] >= MIN_SAMPLES].copy()

    # Convert to minutes for better interpretability in the heatmap.
    agg["avg_duration_min"] = (agg["duration_s"] / agg["n_flights"]) / 60.0

    # Compute cell centres.
    agg["lon_center"] = (agg["cell_lon"] + 0.5) * CELL_SIZE
    agg["lat_center"] = (agg["cell_lat"] + 0.5) * CELL_SIZE


# ========================= Figure =========================

    fig = go.Figure()

    try:
        # Buildings
        geom = airport_geom(airports["LSZH"], "shape")
        if geom is not None:
            add_geom(
                fig,
                geom,
                name="Buildings",
                fill_colour="rgba(0,0,0,1.0)",
                opacity=1.0,
                showlegend=False,
            )

        # Apron
        geom = airport_geom(airports["LSZH"], "apron")
        if geom is not None:
            add_geom(
                fig,
                geom,
                name="Apron",
                line_width=0,
                fill_colour="rgba(255,255,255,1.0)",
                opacity=0.7,
                showlegend=False,
            )

        # Taxiways
        geom = airport_geom(airports["LSZH"], "taxiways")
        if geom is not None:
            add_geom(
                fig,
                geom,
                name="Taxiways",
                line_width=6,
                line_colour="rgba(180,180,180,1.0)",
                opacity=0.5,
                showlegend=False,
            )

        # Runways
        geom = airport_geom(airports["LSZH"], "runways")
        if geom is not None:
            add_geom(
                fig,
                geom,
                name="Runways",
                line_width=6,
                line_colour="black",
                opacity=1.0,
                showlegend=False,
            )

    except Exception:
        pass

    fig.add_trace(
        go.Scatter(
            mode="markers",
            x=agg["lon_center"],
            y=agg["lat_center"],
            marker=dict(
                size=MARKER_SIZE,
                color=agg["avg_duration_min"],
                colorscale="Plasma",
                colorbar=dict(title="Average holding<br>duration [min]"),
                showscale=True,
            ),
            showlegend=False,
            hovertemplate=(
                "Average holding duration: %{marker.color:.2f} min<br>"
                "Longitude: %{x:.6f}<br>"
                "Latitude: %{y:.6f}<extra></extra>"
            ),
        )
    )

    fig.update_layout(
        title=f"Bottleneck Heatmap RWY {RUNWAY} (< {SPEED_THRESHOLD} kt)",
        width=1000,
        height=1000,
        template="plotly_white",
        showlegend=False,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
    )

    # Keep the axes at the same scale so that the airport geometry is not distorted.
    fig.update_yaxes(scaleanchor="x", scaleratio=1)

    fig.show()

### Median-Speed Heatmap
<font color='red'> The desired take-off runway can be entered in the first line of code. </font>

In [ ]:
# ========================= Adjust the parameters here inside =========================

RUNWAY = "28"             # Runway to analyse (28, 32, 16, 10 or 34).

ROUND_DECIMALS = 5        # Controls how strongly nearby Kalman coordinates are merged.
MIN_SAMPLES = 10          # Minimum number of samples (aircraft) required at one snapped location.
MAX_SPEED = 30.0          # [kt] Upper colour limit for the heatmap.
SEGMENT_LENGTH = 0.00005  # [deg] Length of the plotted line segments.
SEGMENT_WIDTH = 5         # Plotly line width of the coloured segments.

# ========================= Adjust the parameters here inside =========================


df = trajs_taxi_out_processed.data.copy()
df = df[df["cut_runway"] == RUNWAY]

# Keep only rows with valid Kalman coordinates and speed values.
df = df.dropna(
    subset=[
        "longitude_kalman",
        "latitude_kalman",
        "groundspeed_kalman",
        "track_kalman",
    ]
)

if df.empty:
    print("No median-speed data found.")
else:
    # Round the snapped Kalman coordinates to merge nearby taxiway positions.
    df["lon_round"] = df["longitude_kalman"].round(ROUND_DECIMALS)
    df["lat_round"] = df["latitude_kalman"].round(ROUND_DECIMALS)

    # Aggregate the median speed, median track, and the number of samples per snapped location.
    agg = (
        df.groupby(["lon_round", "lat_round"])
        .agg(
            median_speed_kt=("groundspeed_kalman", "median"),
            median_track_deg=("track_kalman", "median"),
            n_samples=("groundspeed_kalman", "size"),
        )
        .reset_index()
    )

    # Remove locations with too few samples to reduce visual noise.
    agg = agg[agg["n_samples"] >= MIN_SAMPLES].copy()

    if agg.empty:
        print("No median-speed data found after sample filtering.")
    else:
        # Build short line segments centred on the snapped taxiway locations.
        # The segment orientation follows the local median track direction.
        half_len = SEGMENT_LENGTH / 2.0

        # Track convention:
        # track = degrees(arctan2(dx, dy))
        # Therefore:
        # dx = sin(track), dy = cos(track)
        theta = np.deg2rad(agg["median_track_deg"].to_numpy())

        agg["lon_start"] = agg["lon_round"] - half_len * np.sin(theta)
        agg["lon_end"]   = agg["lon_round"] + half_len * np.sin(theta)
        agg["lat_start"] = agg["lat_round"] - half_len * np.cos(theta)
        agg["lat_end"]   = agg["lat_round"] + half_len * np.cos(theta)

        # Map the median speed values to actual Plotly colours.
        colour_scale = "YlOrRd_r"

        speed_norm = np.clip(agg["median_speed_kt"] / MAX_SPEED, 0, 1)
        agg["segment_colour"] = [
            sample_colorscale(colour_scale, [v])[0]
            for v in speed_norm
        ]


# ========================= Figure =========================

        fig = go.Figure()

        try:
            # Buildings
            geom = airport_geom(airports["LSZH"], "shape")
            if geom is not None:
                add_geom(
                    fig,
                    geom,
                    name="Buildings",
                    fill_colour="rgba(0,0,0,1.0)",
                    opacity=1.0,
                    showlegend=False,
                )

            # Apron
            geom = airport_geom(airports["LSZH"], "apron")
            if geom is not None:
                add_geom(
                    fig,
                    geom,
                    name="Apron",
                    line_width=0,
                    fill_colour="rgba(255,255,255,1.0)",
                    opacity=0.7,
                    showlegend=False,
                )

            # Taxiways
            geom = airport_geom(airports["LSZH"], "taxiways")
            if geom is not None:
                add_geom(
                    fig,
                    geom,
                    name="Taxiways",
                    line_width=6,
                    line_colour="rgba(180,180,180,1.0)",
                    opacity=0.5,
                    showlegend=False,
                )

            # Runways
            geom = airport_geom(airports["LSZH"], "runways")
            if geom is not None:
                add_geom(
                    fig,
                    geom,
                    name="Runways",
                    line_width=6,
                    line_colour="black",
                    opacity=1.0,
                    showlegend=False,
                )

        except Exception:
            pass

        # Add coloured line segments location by location.
        for _, row in agg.iterrows():
            fig.add_trace(
                go.Scatter(
                    mode="lines",
                    x=[row["lon_start"], row["lon_end"]],
                    y=[row["lat_start"], row["lat_end"]],
                    line=dict(
                        width=SEGMENT_WIDTH,
                        color=row["segment_colour"],
                    ),
                    showlegend=False,
                    hovertemplate=(
                        f"Median speed: {row['median_speed_kt']:.2f} kt<br>"
                        f"Samples: {int(row['n_samples'])}<br>"
                        f"Longitude: {row['lon_round']:.6f}<br>"
                        f"Latitude: {row['lat_round']:.6f}<extra></extra>"
                    ),
                )
            )

        # Add an invisible marker trace only to display the shared colour bar.
        fig.add_trace(
            go.Scatter(
                mode="markers",
                x=agg["lon_round"],
                y=agg["lat_round"],
                marker=dict(
                    size=0.01,
                    color=agg["median_speed_kt"],
                    colorscale=colour_scale,
                    cmin=0,
                    cmax=MAX_SPEED,
                    colorbar=dict(title="Median speed [kt]"),
                    showscale=True,
                ),
                opacity=0,
                hoverinfo="skip",
                showlegend=False,
            )
        )

        fig.update_layout(
            title=f"Median-Speed Heatmap RWY {RUNWAY}",
            width=1000,
            height=1000,
            template="plotly_white",
            showlegend=False,
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
        )

        # Keep the axes at the same scale so that the airport geometry is not distorted.
        fig.update_yaxes(scaleanchor="x", scaleratio=1)

        fig.show()

In [ ]:
# ========================= Adjust the parameters here inside =========================

RUNWAY = "28"             # Runway to analyse (28, 32, 16, 10 or 34).

ROUND_DECIMALS = 5        # Controls how strongly nearby Kalman coordinates are merged.
MIN_SAMPLES = 10          # Minimum number of samples (aircraft) required at one snapped location.
MAX_SPEED = 30.0          # [kt] Upper colour limit for the heatmap.
MARKER_SIZE = 5           # Plotly marker size of the coloured circles.

# ========================= Adjust the parameters here inside =========================


df = trajs_taxi_out_processed.data.copy()
df = df[df["cut_runway"] == RUNWAY]

# Keep only rows with valid Kalman coordinates and speed values.
df = df.dropna(
    subset=[
        "longitude_kalman",
        "latitude_kalman",
        "groundspeed_kalman",
    ]
)

if df.empty:
    print("No median-speed data found.")
else:
    # Round the snapped Kalman coordinates to merge nearby taxiway positions.
    df["lon_round"] = df["longitude_kalman"].round(ROUND_DECIMALS)
    df["lat_round"] = df["latitude_kalman"].round(ROUND_DECIMALS)

    # Aggregate the median speed and the number of samples per snapped location.
    agg = (
        df.groupby(["lon_round", "lat_round"])
        .agg(
            median_speed_kt=("groundspeed_kalman", "median"),
            n_samples=("groundspeed_kalman", "size"),
        )
        .reset_index()
    )

    # Remove locations with too few samples to reduce visual noise.
    agg = agg[agg["n_samples"] >= MIN_SAMPLES].copy()

    if agg.empty:
        print("No median-speed data found after sample filtering.")
    else:
        # Map the median speed values to actual Plotly colours.
        colour_scale = "YlOrRd_r"


# ========================= Figure =========================

        fig = go.Figure()

        try:
            # Buildings
            geom = airport_geom(airports["LSZH"], "shape")
            if geom is not None:
                add_geom(
                    fig,
                    geom,
                    name="Buildings",
                    fill_colour="rgba(0,0,0,1.0)",
                    opacity=1.0,
                    showlegend=False,
                )

            # Apron
            geom = airport_geom(airports["LSZH"], "apron")
            if geom is not None:
                add_geom(
                    fig,
                    geom,
                    name="Apron",
                    line_width=0,
                    fill_colour="rgba(255,255,255,1.0)",
                    opacity=0.7,
                    showlegend=False,
                )

            # Taxiways
            geom = airport_geom(airports["LSZH"], "taxiways")
            if geom is not None:
                add_geom(
                    fig,
                    geom,
                    name="Taxiways",
                    line_width=6,
                    line_colour="rgba(180,180,180,1.0)",
                    opacity=0.5,
                    showlegend=False,
                )

            # Runways
            geom = airport_geom(airports["LSZH"], "runways")
            if geom is not None:
                add_geom(
                    fig,
                    geom,
                    name="Runways",
                    line_width=6,
                    line_colour="black",
                    opacity=1.0,
                    showlegend=False,
                )

        except Exception:
            pass

        # Add coloured circles location by location.
        fig.add_trace(
            go.Scatter(
                mode="markers",
                x=agg["lon_round"],
                y=agg["lat_round"],
                marker=dict(
                    size=MARKER_SIZE,
                    color=agg["median_speed_kt"],
                    colorscale=colour_scale,
                    cmin=0,
                    cmax=MAX_SPEED,
                    colorbar=dict(title="Median speed [kt]"),
                    showscale=True,
                ),
                customdata=agg[["n_samples"]].to_numpy(),
                showlegend=False,
                hovertemplate=(
                    "Median speed: %{marker.color:.2f} kt<br>"
                    "Samples: %{customdata[0]}<br>"
                    "Longitude: %{x:.6f}<br>"
                    "Latitude: %{y:.6f}<extra></extra>"
                ),
            )
        )

        fig.update_layout(
            title=f"Median-Speed Heatmap RWY {RUNWAY}",
            width=1000,
            height=1000,
            template="plotly_white",
            showlegend=False,
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
        )

        # Keep the axes at the same scale so that the airport geometry is not distorted.
        fig.update_yaxes(scaleanchor="x", scaleratio=1)

        fig.show()

In [ ]:
# ========================= Adjust the parameters here inside =========================

RUNWAY = "28"              # Runway to analyse (28, 32, 16, 10 or 34).

ROUND_DECIMALS = 5         # Controls how strongly nearby Kalman coordinates are merged.
MIN_SAMPLES = 10           # Minimum number of samples (aircraft) required at one snapped location.
MARKER_SIZE = 5            # Plotly marker size of the coloured circles.

REFERENCE_PERCENTILE = 90  # Reference percentile, e.g. 90 for the fastest tenth of speeds.
BASELINE_PERCENTILE = 50   # Baseline percentile, e.g. 50 for the median.

# ========================= Adjust the parameters here inside =========================


df = trajs_taxi_out_processed.data.copy()
df = df[df["cut_runway"] == RUNWAY]

# Keep only rows with valid Kalman coordinates and speed values.
df = df.dropna(
    subset=[
        "longitude_kalman",
        "latitude_kalman",
        "groundspeed_kalman",
    ]
)

if df.empty:
    print("No percentile-difference data found.")
else:
    # Round the snapped Kalman coordinates to merge nearby taxiway positions.
    df["lon_round"] = df["longitude_kalman"].round(ROUND_DECIMALS)
    df["lat_round"] = df["latitude_kalman"].round(ROUND_DECIMALS)

    # Aggregate the selected percentile speeds and the number of samples per snapped location.
    agg = (
        df.groupby(["lon_round", "lat_round"])
        .agg(
            reference_speed_kt=(
                "groundspeed_kalman",
                lambda s: s.quantile(REFERENCE_PERCENTILE / 100),
            ),
            baseline_speed_kt=(
                "groundspeed_kalman",
                lambda s: s.quantile(BASELINE_PERCENTILE / 100),
            ),
            n_samples=("groundspeed_kalman", "size"),
        )
        .reset_index()
    )

    # Calculate the difference between the higher reference percentile and the baseline percentile.
    agg["speed_diff_kt"] = agg["reference_speed_kt"] - agg["baseline_speed_kt"]

    # Remove locations with too few samples to reduce visual noise.
    agg = agg[agg["n_samples"] >= MIN_SAMPLES].copy()

    if agg.empty:
        print("No percentile-difference data found after sample filtering.")
    else:
        # Map larger speed differences to red and smaller ones to yellow.
        colour_scale = "YlOrRd"

        # Let the upper colour limit adapt to the available data.
        max_speed_diff = agg["speed_diff_kt"].max()


# ========================= Figure =========================

        fig = go.Figure()

        try:
            # Buildings
            geom = airport_geom(airports["LSZH"], "shape")
            if geom is not None:
                add_geom(
                    fig,
                    geom,
                    name="Buildings",
                    fill_colour="rgba(0,0,0,1.0)",
                    opacity=1.0,
                    showlegend=False,
                )

            # Apron
            geom = airport_geom(airports["LSZH"], "apron")
            if geom is not None:
                add_geom(
                    fig,
                    geom,
                    name="Apron",
                    line_width=0,
                    fill_colour="rgba(255,255,255,1.0)",
                    opacity=0.7,
                    showlegend=False,
                )

            # Taxiways
            geom = airport_geom(airports["LSZH"], "taxiways")
            if geom is not None:
                add_geom(
                    fig,
                    geom,
                    name="Taxiways",
                    line_width=6,
                    line_colour="rgba(180,180,180,1.0)",
                    opacity=0.5,
                    showlegend=False,
                )

            # Runways
            geom = airport_geom(airports["LSZH"], "runways")
            if geom is not None:
                add_geom(
                    fig,
                    geom,
                    name="Runways",
                    line_width=6,
                    line_colour="black",
                    opacity=1.0,
                    showlegend=False,
                )

        except Exception:
            pass

        # Add coloured circles location by location.
        fig.add_trace(
            go.Scatter(
                mode="markers",
                x=agg["lon_round"],
                y=agg["lat_round"],
                marker=dict(
                    size=MARKER_SIZE,
                    color=agg["speed_diff_kt"],
                    colorscale=colour_scale,
                    cmin=0,
                    cmax=max_speed_diff,
                    colorbar=dict(
                        title=f"p{REFERENCE_PERCENTILE} – p{BASELINE_PERCENTILE} speed [kt]"
                    ),
                    showscale=True,
                ),
                customdata=agg[
                    ["reference_speed_kt", "baseline_speed_kt", "n_samples"]
                ].to_numpy(),
                showlegend=False,
                hovertemplate=(
                    f"p{REFERENCE_PERCENTILE} speed: "
                    "%{customdata[0]:.2f} kt<br>"
                    f"p{BASELINE_PERCENTILE} speed: "
                    "%{customdata[1]:.2f} kt<br>"
                    "Difference: %{marker.color:.2f} kt<br>"
                    "Samples: %{customdata[2]}<br>"
                    "Longitude: %{x:.6f}<br>"
                    "Latitude: %{y:.6f}<extra></extra>"
                ),
            )
        )

        fig.update_layout(
            title=f"Speed-Difference Heatmap RWY {RUNWAY} (p{REFERENCE_PERCENTILE} – p{BASELINE_PERCENTILE})",
            width=1000,
            height=1000,
            template="plotly_white",
            showlegend=False,
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
        )

        # Keep the axes at the same scale so that the airport geometry is not distorted.
        fig.update_yaxes(scaleanchor="x", scaleratio=1)

        fig.show()

## Step 6: Feature Engineering
In the following cells, features for our ML model are engineered from the identified bottlenecks (heatmaps). These features are then exported as a parquet file (`helper_bottleneck_features.parquet`).

#### <font color='red'> CHANGE THE GLOBAL PROCESSING PARAMETERS IN THE CELL BELOW: </font>

In [ ]:
MIN_POINTS_IN_SEGMENT = 5     # Minimum number of trajectory points that must lie inside a segment before the segment is counted as "used".
                              # 1 trajectorie point = 1 second.

SEGMENT_WIDTH_M = 25.0        # [m] Taxiway segment width.

EARTH_RADIUS_M = 6_371_000.0  # [m] Earth radius used for the local lon/lat <-> x/y conversion.

In the following cell:
- A local reference coordinate is derived automatically or estimated from the data.
- The coordinate and timestamp columns used for the feature engineering are defined.
- All manual taxiway segments are specified.

In [ ]:
# The source object from the previous code.
SOURCE_TRAJS = trajs_taxi_out_processed

# Coordinate columns used for the segment checks.
LON_COL = "longitude_kalman"
LAT_COL = "latitude_kalman"

# Time column used to derive taxi_start and taxi_end per flight (flight_id).
TIME_COL = "timestamp"


# Optional: automatically derive a local reference point from the airport definition if available.
try:
    REF_LON, REF_LAT = airport_lon_lat(airports["LSZH"])
except Exception:
    # Fallback: use the mean Kalman position of all processed points.
    _tmp_df = SOURCE_TRAJS.data if hasattr(SOURCE_TRAJS, "data") else SOURCE_TRAJS
    REF_LON = float(_tmp_df[LON_COL].dropna().mean())
    REF_LAT = float(_tmp_df[LAT_COL].dropna().mean())


# Manual taxiway segment definitions:
segment_definitions = [
    {
        "segment_name": "A1",
        "start_lon": 8.553859,
        "start_lat": 47.456705,
        "end_lon":   8.558005,
        "end_lat":   47.456406,
    },
    {
        "segment_name": "A2",
        "start_lon": 8.559059,
        "start_lat": 47.456345,
        "end_lon":   8.562211,
        "end_lat":   47.456123,
    },
    {
        "segment_name": "A3",
        "start_lon": 8.563042,
        "start_lat": 47.456068,
        "end_lon":   8.564326,
        "end_lat":   47.455972,
    },
    {
        "segment_name": "A4",
        "start_lon": 8.564738,
        "start_lat": 47.455947,
        "end_lon":   8.569491,
        "end_lat":   47.455606,
    },
    {
        "segment_name": "A5",
        "start_lon": 8.570187,
        "start_lat": 47.456345,
        "end_lon":   8.570125,
        "end_lat":   47.455816,
    },
    {
        "segment_name": "B1",
        "start_lon": 8.570240,
        "start_lat": 47.456897,
        "end_lon":   8.570251,
        "end_lat":   47.457445,
    },
    {
        "segment_name": "B2",
        "start_lon": 8.569904,
        "start_lat": 47.457672,
        "end_lon":   8.565811,
        "end_lat":   47.457977,
    },
    {
        "segment_name": "B3",
        "start_lon": 8.565060,
        "start_lat": 47.458034,
        "end_lon":   8.563363,
        "end_lat":   47.458159,
    },
    {
        "segment_name": "B4",
        "start_lon": 8.562422,
        "start_lat": 47.458222,
        "end_lon":   8.559506,
        "end_lat":   47.458439,
    },
    {
        "segment_name": "B5",
        "start_lon": 8.558455,
        "start_lat": 47.458503,
        "end_lon":   8.555844,
        "end_lat":   47.458702,
    },
    {
        "segment_name": "B6",
        "start_lon": 8.555161,
        "start_lat": 47.458745,
        "end_lon":   8.552581,
        "end_lat":   47.458922,
    },
    {
        "segment_name": "B7",
        "start_lon": 8.542747,
        "start_lat": 47.459494,
        "end_lon":   8.542826,
        "end_lat":   47.458883,
    },
    {
        "segment_name": "B8",
        "start_lon": 8.549445,
        "start_lat": 47.459146,
        "end_lon":   8.548184,
        "end_lat":   47.459238,
    },
    {
        "segment_name": "B9",
        "start_lon": 8.537807,
        "start_lat": 47.459196,
        "end_lon":   8.538028,
        "end_lat":   47.459775,
    },
    {
        "segment_name": "B10",
        "start_lon": 8.547963,
        "start_lat": 47.459249,
        "end_lon":   8.543140,
        "end_lat":   47.459597,
    },
    {
        "segment_name": "B11",
        "start_lon": 8.542509,
        "start_lat": 47.459636,
        "end_lon":   8.538317,
        "end_lat":   47.459931,
    },
    {
        "segment_name": "C1",
        "start_lon": 8.560201,
        "start_lat": 47.459281,
        "end_lon":   8.562516,
        "end_lat":   47.459120,
    },
    {
        "segment_name": "C2",
        "start_lon": 8.560206,
        "start_lat": 47.459726,
        "end_lon":   8.562590,
        "end_lat":   47.459557,
    },
    {
        "segment_name": "C3",
        "start_lon": 8.560424,
        "start_lat": 47.460161,
        "end_lon":   8.562553,
        "end_lat":   47.460006,
    },
    {
        "segment_name": "C4",
        "start_lon": 8.551931,
        "start_lat": 47.459862,
        "end_lon":   8.555330,
        "end_lat":   47.459620,
    },
    {
        "segment_name": "C5",
        "start_lon": 8.555950,
        "start_lat": 47.459574,
        "end_lon":   8.559029,
        "end_lat":   47.459361,
    },
    {
        "segment_name": "D1",
        "start_lon": 8.551000,
        "start_lat": 47.462836,
        "end_lon":   8.553445,
        "end_lat":   47.462662,
    },
    {
        "segment_name": "D2",
        "start_lon": 8.553555,
        "start_lat": 47.462655,
        "end_lon":   8.558488,
        "end_lat":   47.462306,
    },
    {
        "segment_name": "E1",
        "start_lon": 8.537206,
        "start_lat": 47.477064,
        "end_lon":   8.536332,
        "end_lat":   47.475998,
    },
    {
        "segment_name": "E2",
        "start_lon": 8.536701,
        "start_lat": 47.475772,
        "end_lon":   8.538005,
        "end_lat":   47.476099,
    },
    {
        "segment_name": "E3",
        "start_lon": 8.542749,
        "start_lat": 47.469654,
        "end_lon":   8.541904,
        "end_lat":   47.470181,
    },
    {
        "segment_name": "E4",
        "start_lon": 8.545831,
        "start_lat": 47.464803,
        "end_lon":   8.544345,
        "end_lat":   47.464328,
    },
    {
        "segment_name": "E5",
        "start_lon": 8.546785,
        "start_lat": 47.460998,
        "end_lon":   8.548755,
        "end_lat":   47.460154,
    },
    {
        "segment_name": "E6",
        "start_lon": 8.551654,
        "start_lat": 47.456319,
        "end_lon":   8.550945,
        "end_lat":   47.455002,
    },
    {
        "segment_name": "E7",
        "start_lon": 8.551530,
        "start_lat": 47.454217,
        "end_lon":   8.553228,
        "end_lat":   47.453701,
    },
    {
        "segment_name": "E8",
        "start_lon": 8.554842,
        "start_lat": 47.449075,
        "end_lon":   8.556423,
        "end_lat":   47.449569,
    },
    {
        "segment_name": "E9",
        "start_lon": 8.557054,
        "start_lat": 47.445912,
        "end_lon":   8.558496,
        "end_lat":   47.446345,
    },
    {
        "segment_name": "E10",
        "start_lon": 8.537869,
        "start_lat": 47.477087,
        "end_lon":   8.538573,
        "end_lat":   47.476134,
    },
    {
        "segment_name": "E11",
        "start_lon": 8.538747,
        "start_lat": 47.475858,
        "end_lon":   8.542998,
        "end_lat":   47.469721,
    },
    {
        "segment_name": "E12",
        "start_lon": 8.541573,
        "start_lat": 47.470236,
        "end_lon":   8.540554,
        "end_lat":   47.470095,
    },
    {
        "segment_name": "E13",
        "start_lon": 8.543188,
        "start_lat": 47.469423,
        "end_lon":   8.543985,
        "end_lat":   47.468389,
    },
    {
        "segment_name": "E14",
        "start_lon": 8.543985,
        "start_lat": 47.468266,
        "end_lon":   8.544447,
        "end_lat":   47.467421,
    },
    {
        "segment_name": "E15",
        "start_lon": 8.544570,
        "start_lat": 47.467258,
        "end_lon":   8.546040,
        "end_lat":   47.465078,
    },
    {
        "segment_name": "E16",
        "start_lon": 8.546265,
        "start_lat": 47.464729,
        "end_lon":   8.547310,
        "end_lat":   47.463304,
    },
    {
        "segment_name": "E17",
        "start_lon": 8.547786,
        "start_lat": 47.462620,
        "end_lon":   8.549055,
        "end_lat":   47.460559,
    },
    {
        "segment_name": "E18",
        "start_lon": 8.550431,
        "start_lat": 47.458795,
        "end_lon":   8.551632,
        "end_lat":   47.457021,
    },
    {
        "segment_name": "E19",
        "start_lon": 8.552310,
        "start_lat": 47.456003,
        "end_lon":   8.552656,
        "end_lat":   47.455521,
    },
    {
        "segment_name": "E20",
        "start_lon": 8.553077,
        "start_lat": 47.454901,
        "end_lon":   8.553607,
        "end_lat":   47.454125,
    },
    {
        "segment_name": "E21",
        "start_lon": 8.555137,
        "start_lat": 47.451912,
        "end_lon":   8.556553,
        "end_lat":   47.449859,
    },
    {
        "segment_name": "E22",
        "start_lon": 8.557099,
        "start_lat": 47.449031,
        "end_lon":   8.558585,
        "end_lat":   47.446863,
    },
    {
        "segment_name": "E23",
        "start_lon": 8.559077,
        "start_lat": 47.446134,
        "end_lon":   8.561080,
        "end_lat":   47.443468,
    },
    {
        "segment_name": "E24",
        "start_lon": 8.561781,
        "start_lat": 47.442537,
        "end_lon":   8.562989,
        "end_lat":   47.440772,
    },
    {
        "segment_name": "F1",
        "start_lon": 8.547876,
        "start_lat": 47.464520,
        "end_lon":   8.546008,
        "end_lat":   47.467231,
    },
    {
        "segment_name": "F2",
        "start_lon": 8.548171,
        "start_lat": 47.465145,
        "end_lon":   8.547091,
        "end_lat":   47.466717,
    },
    {
        "segment_name": "F3",
        "start_lon": 8.548589,
        "start_lat": 47.465561,
        "end_lon":   8.547936,
        "end_lat":   47.466503,
    },
    {
        "segment_name": "F4",
        "start_lon": 8.547797,
        "start_lat": 47.466587,
        "end_lon":   8.545659,
        "end_lat":   47.467600,
    },
    {
        "segment_name": "F5",
        "start_lon": 8.545169,
        "start_lat": 47.467844,
        "end_lon":   8.544251,
        "end_lat":   47.468279,
    },
    {
        "segment_name": "F6",
        "start_lon": 8.548162,
        "start_lat": 47.464531,
        "end_lon":   8.548604,
        "end_lat":   47.465454,
    },
    {
        "segment_name": "F7",
        "start_lon": 8.549483,
        "start_lat": 47.462193,
        "end_lon":   8.550597,
        "end_lat":   47.460549,
    },
    {
        "segment_name": "F8",
        "start_lon": 8.551976,
        "start_lat": 47.458594,
        "end_lon":   8.552951,
        "end_lat":   47.457142,
    },
    {
        "segment_name": "F9",
        "start_lon": 8.553379,
        "start_lat": 47.456229,
        "end_lon":   8.553907,
        "end_lat":   47.455466,
    },
    {
        "segment_name": "F10",
        "start_lon": 8.554422,
        "start_lat": 47.454716,
        "end_lon":   8.555301,
        "end_lat":   47.453416,
    },
    {
        "segment_name": "F11",
        "start_lon": 8.555839,
        "start_lat": 47.452266,
        "end_lon":   8.557275,
        "end_lat":   47.450174,
    },
    {
        "segment_name": "G",
        "start_lon": 8.537917,
        "start_lat": 47.477670,
        "end_lon":   8.539658,
        "end_lat":   47.478820,
    },
    {
        "segment_name": "H1",
        "start_lon": 8.557478,
        "start_lat": 47.463985,
        "end_lon":   8.556718,
        "end_lat":   47.466275,
    },
    {
        "segment_name": "H2",
        "start_lon": 8.559416,
        "start_lat": 47.464287,
        "end_lon":   8.559818,
        "end_lat":   47.462909,
    },
    {
        "segment_name": "H3",
        "start_lon": 8.563354,
        "start_lat": 47.460838,
        "end_lon":   8.563445,
        "end_lat":   47.461480,
    },
    {
        "segment_name": "H4",
        "start_lon": 8.558856,
        "start_lat": 47.462859,
        "end_lon":   8.557612,
        "end_lat":   47.463820,
    },
    {
        "segment_name": "H5",
        "start_lon": 8.563616,
        "start_lat": 47.460882,
        "end_lon":   8.564056,
        "end_lat":   47.461179,
    },
    {
        "segment_name": "H6",
        "start_lon": 8.560859,
        "start_lat": 47.461426,
        "end_lon":   8.562087,
        "end_lat":   47.460523,
    },
    {
        "segment_name": "H7",
        "start_lon": 8.564174,
        "start_lat": 47.458960,
        "end_lon":   8.565188,
        "end_lat":   47.458238,
    },
    {
        "segment_name": "INNER1",
        "start_lon": 8.554864,
        "start_lat": 47.455268,
        "end_lon":   8.557884,
        "end_lat":   47.455051,
    },
    {
        "segment_name": "INNER2",
        "start_lon": 8.558744,
        "start_lat": 47.454993,
        "end_lon":   8.561953,
        "end_lat":   47.454765,
    },
    {
        "segment_name": "INNER3",
        "start_lon": 8.562776,
        "start_lat": 47.455068,
        "end_lon":   8.564120,
        "end_lat":   47.455740,
    },
    {
        "segment_name": "INNER4",
        "start_lon": 8.562591,
        "start_lat": 47.455907,
        "end_lon":   8.562478,
        "end_lat":   47.455221,
    },
    {
        "segment_name": "J1",
        "start_lon": 8.559930,
        "start_lat": 47.461291,
        "end_lon":   8.559713,
        "end_lat":   47.459769,
    },
    {
        "segment_name": "J2",
        "start_lon": 8.559362,
        "start_lat": 47.459094,
        "end_lon":   8.559134,
        "end_lat":   47.458653,
    },
    {
        "segment_name": "J3",
        "start_lon": 8.558821,
        "start_lat": 47.458115,
        "end_lon":   8.558587,
        "end_lat":   47.456636,
    },
    {
        "segment_name": "J4",
        "start_lon": 8.558526,
        "start_lat": 47.456209,
        "end_lon":   8.558382,
        "end_lat":   47.455270,
    },
    {
        "segment_name": "K1",
        "start_lon": 8.562909,
        "start_lat": 47.457920,
        "end_lon":   8.562654,
        "end_lat":   47.456284,
    },
    {
        "segment_name": "L1",
        "start_lon": 8.542222,
        "start_lat": 47.457292,
        "end_lon":   8.542127,
        "end_lat":   47.456549,
    },
    {
        "segment_name": "L2",
        "start_lon": 8.537935,
        "start_lat": 47.457249,
        "end_lon":   8.541849,
        "end_lat":   47.456340,
    },
    {
        "segment_name": "L3",
        "start_lon": 8.542569,
        "start_lat": 47.456183,
        "end_lon":   8.547224,
        "end_lat":   47.455146,
    },
    {
        "segment_name": "L4",
        "start_lon": 8.548369,
        "start_lat": 47.454887,
        "end_lon":   8.549824,
        "end_lat":   47.454542,
    },
    {
        "segment_name": "L7",
        "start_lon": 8.544455,
        "start_lat": 47.458052,
        "end_lon":   8.542448,
        "end_lat":   47.457441,
    },
    {
        "segment_name": "L9",
        "start_lon": 8.537772,
        "start_lat": 47.458634,
        "end_lon":   8.537651,
        "end_lat":   47.457569,
    },
    {
        "segment_name": "Link4",
        "start_lon": 8.555620,
        "start_lat": 47.459410,
        "end_lon":   8.555545,
        "end_lat":   47.458901,
    },
    {
        "segment_name": "Link5",
        "start_lon": 8.547907,
        "start_lat": 47.462946,
        "end_lon":   8.548613,
        "end_lat":   47.463138,
    },
    {
        "segment_name": "Link6",
        "start_lon": 8.547566,
        "start_lat": 47.463381,
        "end_lon":   8.547898,
        "end_lat":   47.464096,
    },
    {
        "segment_name": "Link7",
        "start_lon": 8.545159,
        "start_lat": 47.467659,
        "end_lon":   8.544751,
        "end_lat":   47.467396,
    },
    {
        "segment_name": "M1",
        "start_lon": 8.558346,
        "start_lat": 47.449502,
        "end_lon":   8.559113,
        "end_lat":   47.449444,
    },
    {
        "segment_name": "M2",
        "start_lon": 8.559446,
        "start_lat": 47.449262,
        "end_lon":   8.560744,
        "end_lat":   47.447367,
    },
    {
        "segment_name": "M3",
        "start_lon": 8.560825,
        "start_lat": 47.447254,
        "end_lon":   8.562262,
        "end_lat":   47.445168,
    },
    {
        "segment_name": "M4",
        "start_lon": 8.562836,
        "start_lat": 47.445009,
        "end_lon":   8.564925,
        "end_lat":   47.445664,
    },
    {
        "segment_name": "M5",
        "start_lon": 8.565500,
        "start_lat": 47.445568,
        "end_lon":   8.566241,
        "end_lat":   47.444502,
    },
    {
        "segment_name": "M6",
        "start_lon": 8.566107,
        "start_lat": 47.444135,
        "end_lon":   8.562277,
        "end_lat":   47.442989,
    },
    {
        "segment_name": "N1",
        "start_lon": 8.556077,
        "start_lat": 47.452730,
        "end_lon":   8.558145,
        "end_lat":   47.452584,
    },
    {
        "segment_name": "N2",
        "start_lon": 8.558233,
        "start_lat": 47.452577,
        "end_lon":   8.560260,
        "end_lat":   47.452431,
    },
    {
        "segment_name": "P1",
        "start_lon": 8.550619,
        "start_lat": 47.463166,
        "end_lon":   8.550667,
        "end_lat":   47.463516,
    },
    {
        "segment_name": "P2",
        "start_lon": 8.550611,
        "start_lat": 47.463649,
        "end_lon":   8.549857,
        "end_lat":   47.464745,
    },
    {
        "segment_name": "R1",
        "start_lon": 8.547817,
        "start_lat": 47.454599,
        "end_lon":   8.548148,
        "end_lat":   47.453938,
    },
    {
        "segment_name": "R2",
        "start_lon": 8.548453,
        "start_lat": 47.453430,
        "end_lon":   8.549215,
        "end_lat":   47.452581,
    },
    {
        "segment_name": "R3",
        "start_lon": 8.549819,
        "start_lat": 47.451366,
        "end_lon":   8.551117,
        "end_lat":   47.449483,
    },
    {
        "segment_name": "R7",
        "start_lon": 8.549882,
        "start_lat": 47.452716,
        "end_lon":   8.550329,
        "end_lat":   47.453583,
    },
    {
        "segment_name": "R8",
        "start_lon": 8.549388,
        "start_lat": 47.447344,
        "end_lon":   8.553928,
        "end_lat":   47.448786,
    },
    {
        "segment_name": "S1",
        "start_lon": 8.548963,
        "start_lat": 47.447315,
        "end_lon":   8.547849,
        "end_lat":   47.447553,
    },
    {
        "segment_name": "S2",
        "start_lon": 8.547483,
        "start_lat": 47.447654,
        "end_lon":   8.549276,
        "end_lat":   47.451366,
    },
    {
        "segment_name": "T1",
        "start_lon": 8.545513,
        "start_lat": 47.450808,
        "end_lon":   8.546918,
        "end_lat":   47.453198,
    },
    {
        "segment_name": "T2",
        "start_lon": 8.547086,
        "start_lat": 47.453301,
        "end_lon":   8.547942,
        "end_lat":   47.453567,
    },
    {
        "segment_name": "Z1",
        "start_lon": 8.572420,
        "start_lat": 47.457352,
        "end_lon":   8.570676,
        "end_lat":   47.457473,
    },
]

segments_df = pd.DataFrame(segment_definitions)

In the following cell:
- Helper functions for coordinate conversion (lat/lon -> x/y) are created.
- Rectangular polygons are generated for all manual taxiway segments.
- Taxiway segment lengths are calculated in metres (`seg_length_m`).
- `taxi_start` and `taxi_end` are derived for each flight (`flight_id`) from the trajectory timestamps (`timestamp`).
- Flights are checked for taxiway segment usage based on trajectory points and binary (1/0) taxiway segment usage features are created for each flight (`flight_id`).
- Historical taxiway segment point counts are collected.
- The helper table is sorted chronologically by `taxi_start`.

In [ ]:
def lonlat_to_local_xy(lon, lat, lon0=REF_LON, lat0=REF_LAT, radius=EARTH_RADIUS_M):
    """
    Converts longitude and latitude coordinates to a local Cartesian x/y system.

    Uses an equirectangular approximation around a local reference point so that
    distances can be treated consistently in metres.

    Args:
        lon: Longitude value or array of longitude values.
        lat: Latitude value or array of latitude values.
        lon0: Reference longitude of the local origin.
        lat0: Reference latitude of the local origin.
        radius: Earth radius used for the projection.

    Returns:
        tuple: Local x and y coordinates in metres.
    """
    lon_rad = np.radians(lon)
    lat_rad = np.radians(lat)
    lon0_rad = np.radians(lon0)
    lat0_rad = np.radians(lat0)

    x = radius * (lon_rad - lon0_rad) * np.cos(lat0_rad)
    y = radius * (lat_rad - lat0_rad)

    return x, y


def local_xy_to_lonlat(x, y, lon0=REF_LON, lat0=REF_LAT, radius=EARTH_RADIUS_M):
    """
    Converts local Cartesian x/y coordinates back to longitude and latitude.

    Reverses the local equirectangular transformation based on the selected
    reference point.

    Args:
        x: Local x coordinate or array of x coordinates in metres.
        y: Local y coordinate or array of y coordinates in metres.
        lon0: Reference longitude of the local origin.
        lat0: Reference latitude of the local origin.
        radius: Earth radius used for the projection.

    Returns:
        tuple: Longitude and latitude coordinates.
    """
    lon0_rad = np.radians(lon0)
    lat0_rad = np.radians(lat0)

    lon_rad = lon0_rad + (x / (radius * np.cos(lat0_rad)))
    lat_rad = lat0_rad + (y / radius)

    lon = np.degrees(lon_rad)
    lat = np.degrees(lat_rad)

    return lon, lat


def build_segment_polygon(start_lon, start_lat, end_lon, end_lat, width_m):
    """
    Builds a rectangular polygon around a segment centre line.

    The rectangle is created from the start and end coordinates with a fixed
    width in metres. The segment ends are represented as straight edges and are not curved.

    Args:
        start_lon: Start longitude of the segment.
        start_lat: Start latitude of the segment.
        end_lon: End longitude of the segment.
        end_lat: End latitude of the segment.
        width_m: Total segment width in metres.

    Returns:
        tuple: A Shapely polygon of the segment area and the segment length
        in metres.

    Raises:
        ValueError: If the start and end coordinates are identical.
    """
    x1, y1 = lonlat_to_local_xy(start_lon, start_lat)
    x2, y2 = lonlat_to_local_xy(end_lon, end_lat)

    dx = x2 - x1
    dy = y2 - y1
    length_m = float(np.hypot(dx, dy))

    if length_m == 0.0:
        raise ValueError("Segment start and end coordinates must not be identical.")

    # Unit normal vector to the centre line.
    nx = -dy / length_m
    ny = dx / length_m

    half_width = width_m / 2.0

    # Rectangle corners in local x/y coordinates.
    c1 = (x1 + nx * half_width, y1 + ny * half_width)
    c2 = (x2 + nx * half_width, y2 + ny * half_width)
    c3 = (x2 - nx * half_width, y2 - ny * half_width)
    c4 = (x1 - nx * half_width, y1 - ny * half_width)

    # Convert the corners back to longitude/latitude.
    corners_lonlat = [local_xy_to_lonlat(x, y) for x, y in (c1, c2, c3, c4)]
    polygon = Polygon(corners_lonlat)

    return polygon, length_m


# Build all segment polygons once.
segment_polygons = {}
segment_lengths_m = {}

for row in segments_df.itertuples(index=False):
    polygon, length_m = build_segment_polygon(
        start_lon=row.start_lon,
        start_lat=row.start_lat,
        end_lon=row.end_lon,
        end_lat=row.end_lat,
        width_m=SEGMENT_WIDTH_M,
    )
    segment_polygons[row.segment_name] = polygon
    segment_lengths_m[row.segment_name] = length_m

# Extract the processed trajectory DataFrame.
processed_df = SOURCE_TRAJS.data if hasattr(SOURCE_TRAJS, "data") else SOURCE_TRAJS

# Keep only the columns needed for the segment checks and time-based rolling features.
work_df = processed_df.loc[:, ["flight_id", TIME_COL, LON_COL, LAT_COL]].dropna(
    subset=["flight_id", TIME_COL, LON_COL, LAT_COL]
)

# Convert the timestamp column to datetime if it's not already, and ensure it's timezone-aware in UTC.
work_df = work_df.assign(
    **{TIME_COL: pd.to_datetime(work_df[TIME_COL], utc=True)}
)

# Ensure that the timestamp column is a proper timezone-aware datetime column.
work_df[TIME_COL] = pd.to_datetime(work_df[TIME_COL], utc=True)

# Derive one taxi_start and taxi_end timestamp per flight.
taxi_times_per_flight = (
    work_df
    .groupby("flight_id", sort=False)[TIME_COL]
    .agg(taxi_start="min", taxi_end="max")
    .reset_index()
)

# Prepare all flight IDs for the final helper table.
all_flight_ids = (
    taxi_times_per_flight["flight_id"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

# Base table with all flight IDs and their taxi time interval.
helper_bottleneck_features = pd.DataFrame({"flight_id": all_flight_ids}).merge(
    taxi_times_per_flight,
    on="flight_id",
    how="left",
)

# Store segment feature columns here first, then concatenate once at the end.
segment_feature_cols = {}

# Store point counts per flight and segment for later rolling historical time statistics.
segment_hit_counts_parts = []

lon_vals = work_df[LON_COL].to_numpy(dtype=float, copy=False)
lat_vals = work_df[LAT_COL].to_numpy(dtype=float, copy=False)

# Test every segment separately.
for segment_name, poly in segment_polygons.items():
    minx, miny, maxx, maxy = poly.bounds

    # Apply a fast bounding-box prefilter before the exact polygon check.
    bbox_mask = (
        (lon_vals >= minx) & (lon_vals <= maxx) &
        (lat_vals >= miny) & (lat_vals <= maxy)
    )

    col_name = f"seg_{segment_name}"

    if not bbox_mask.any():
        segment_feature_cols[col_name] = np.zeros(len(helper_bottleneck_features), dtype=np.uint8)
        continue

    candidate_idx = np.flatnonzero(bbox_mask)
    candidate_points = shapely.points(lon_vals[candidate_idx], lat_vals[candidate_idx])

    inside_candidate = np.asarray(shapely.covers(poly, candidate_points), dtype=bool)

    if not inside_candidate.any():
        segment_feature_cols[col_name] = np.zeros(len(helper_bottleneck_features), dtype=np.uint8)
        continue

    inside_idx = candidate_idx[inside_candidate]

    counts = (
        work_df.iloc[inside_idx]
        .groupby("flight_id", sort=False)
        .size()
    )

    # Save the raw point counts, because 1 point corresponds to 1 second.
    segment_counts_df = counts.rename("n_points_in_segment").reset_index()
    segment_counts_df["segment_name"] = segment_name
    segment_hit_counts_parts.append(segment_counts_df)

    # A segment is marked as used only if the minimum number of points is reached.
    used_flight_ids = counts.index[counts >= MIN_POINTS_IN_SEGMENT]

    segment_feature_cols[col_name] = (
        helper_bottleneck_features["flight_id"]
        .isin(used_flight_ids)
        .to_numpy(dtype=np.uint8)
    )

# Add all binary segment columns in one go.
segment_features_df = pd.DataFrame(segment_feature_cols, index=helper_bottleneck_features.index)
helper_bottleneck_features = pd.concat(
    [helper_bottleneck_features, segment_features_df],
    axis=1,
)

# Sort flights chronologically so that later rolling operations follow the real time order.
helper_bottleneck_features = (
    helper_bottleneck_features
    .sort_values(["taxi_start", "taxi_end", "flight_id"])
    .reset_index(drop=True)
)

# Combine all per-segment point counts into one long table.
if segment_hit_counts_parts:
    segment_hit_counts = pd.concat(segment_hit_counts_parts, ignore_index=True)
else:
    segment_hit_counts = pd.DataFrame(
        columns=["flight_id", "n_points_in_segment", "segment_name"]
    )

In [ ]:
helper_bottleneck_features.head()

In the following cell:
- The airport layout is plotted as a background map.
- All manual taxiway segment polygons are visualised.

In [ ]:
fig = go.Figure()

# Draw airport background layers, if available.
try:
    layers = [
        ("shape", "Buildings", 1.0, "rgba(0, 0, 0, 1.00)", "rgba(0, 0, 0, 1.00)", 1.00),
        ("movement_area", "Apron & taxiways", 1.0, "rgba(180, 180, 180, 0.90)", "rgba(220, 220, 220, 0.60)", 0.95),
        ("runways", "Runways", 3, "rgba(20, 20, 20, 0.95)", "rgba(40, 40, 40, 0.20)", 0.95),
    ]

    for kind, name, line_width, line_colour, fill_colour, opacity in layers:
        geom = airport_geom(airports["LSZH"], kind)
        if geom is not None:
            add_geom(
                fig,
                geom,
                name=name,
                line_width=line_width,
                line_colour=line_colour,
                fill_colour=fill_colour,
                opacity=opacity,
                showlegend=True,
            )
except Exception:
    pass


# Draw all manual segment polygons and labels.
polygon_legend_added = False

for row in segments_df.itertuples(index=False):
    segment_name = row.segment_name
    poly = segment_polygons[segment_name]

    # Draw the rectangular segment polygon.
    # No border line is used so that the visual width is not artificially enlarged.
    add_geom(
        fig,
        poly,
        name="Manual segment polygon",
        line_width=0,
        fill_colour="green",
        opacity=1.0,
        showlegend=not polygon_legend_added,
    )
    polygon_legend_added = True

    # Place the segment name at the midpoint of the centre line.
    mid_lon = (row.start_lon + row.end_lon) / 2.0
    mid_lat = (row.start_lat + row.end_lat) / 2.0

    fig.add_trace(
        go.Scatter(
            mode="text",
            x=[mid_lon],
            y=[mid_lat],
            text=[f"<b>{segment_name}</b>"],
            textposition="middle center",
            showlegend=False,
            hoverinfo="skip",
            textfont=dict(
                size=6,
                color="darkred",
            ),
        )
    )

fig.update_layout(
    title=f"Manual Taxiway Segment Polygons (Width = {SEGMENT_WIDTH_M:.1f} m)",
    width=1000,
    height=1000,
    margin=dict(l=20, r=20, t=50, b=20),
    legend=dict(orientation="h"),
    template="plotly_white",
)

# Keep the axes at the same scale so that the airport geometry is not distorted.
fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

In the following cell:
- Global p10, median, and p90 taxiway segment times (`p10_hist_seg_time`, `median_hist_seg_time`, and `p90_hist_seg_time`) are calculated for visualisation and understanding only. These global time statistics are **not** further used to guarantee leakage-free feature engineering.
- Taxiway segment lengths (`seg_length_m`) are assembled.
- All taxiway segment properties are merged into one dataset (`seg_props`).

In [ ]:
# Keep only flights that actually used a segment according to the threshold.
valid_segment_hit_counts = segment_hit_counts.loc[
    segment_hit_counts["n_points_in_segment"] >= MIN_POINTS_IN_SEGMENT
]

# Compute global p10, median, and p90 historical segment times.
# These values are kept only for visualisation and understanding. They must not be used for leakage-free model features.
if not valid_segment_hit_counts.empty:
    seg_time_stats = (
        valid_segment_hit_counts
        .groupby("segment_name")["n_points_in_segment"]
        .quantile([0.10, 0.50, 0.90])
        .unstack()
        .rename(
            columns={
                0.10: "p10_hist_seg_time",
                0.50: "median_hist_seg_time",
                0.90: "p90_hist_seg_time",
            }
        )
        .reset_index()
    )
else:
    seg_time_stats = pd.DataFrame(
        columns=[
            "segment_name",
            "p10_hist_seg_time",
            "median_hist_seg_time",
            "p90_hist_seg_time",
        ]
    )

# Create the segment length table.
seg_length_df = pd.DataFrame(
    {
        "segment_name": list(segment_lengths_m.keys()),
        "seg_length_m": np.round(list(segment_lengths_m.values()), 1),
    }
)

# Merge all segment properties.
seg_props = segments_df.loc[:, ["segment_name"]].merge(
    seg_time_stats,
    on="segment_name",
    how="left",
).merge(
    seg_length_df,
    on="segment_name",
    how="left",
)

# Keep missing global time statistics as NaN. A missing value means that no flight historically used this segment.
seg_props["seg_length_m"] = seg_props["seg_length_m"].fillna(0.0)

In [ ]:
seg_props

In the following cell:
- Leakage-free rolling historical taxiway segment time features are calculated. For each flight (`flight_id`), only historical flights with `taxi_end < taxi_start` of the current flight (`flight_id`) are used.
- Rolling p10, median, and p90 times are assigned only to taxiway segments used by the current flight (`flight_id`).
- Taxiway segment lengths are assigned only to segments used by the current flight (`flight_id`).
- If no valid historical data exists for a used taxiway segment, the corresponding rolling time feature remains `NaN`.
- A flight-level feature DataFrame is constructed (`helper_bottleneck_features`).

In [ ]:
def compute_rolling_segment_quantiles(
    helper_binary,
    valid_segment_hit_counts,
    segments_df,
    segment_lengths_m,
):
    """
    Computes leakage-free rolling historical segment time features.

    For every current flight and segment, the historical distribution is built
    only from flights whose taxi_end is strictly before the current flight's
    taxi_start. Therefore, no future or overlapping flight can contribute to
    the current flight's historical segment statistics.

    Args:
        helper_binary: DataFrame containing flight_id, taxi_start, taxi_end and binary segment columns.
        valid_segment_hit_counts: Long DataFrame with valid segment point counts.
        segments_df: DataFrame containing the segment definitions.
        segment_lengths_m: Dictionary with segment lengths in metres.

    Returns:
        DataFrame: Flight-level table with rolling segment statistics and segment lengths.
    """
    result_cols = {
        "flight_id": helper_binary["flight_id"].to_numpy(copy=False),
        "taxi_start": helper_binary["taxi_start"].to_numpy(copy=False),
        "taxi_end": helper_binary["taxi_end"].to_numpy(copy=False),
    }

    current_start_ns = helper_binary["taxi_start"].astype("int64").to_numpy(copy=False)

    # Attach taxi_end to every historical segment observation.
    history_df = valid_segment_hit_counts.merge(
        helper_binary.loc[:, ["flight_id", "taxi_end"]],
        on="flight_id",
        how="left",
    ).dropna(subset=["taxi_end"])

    if not history_df.empty:
        history_df = history_df.assign(
            taxi_end_ns=history_df["taxi_end"].astype("int64")
        )

    for segment_name in segments_df["segment_name"]:
        seg_col = f"seg_{segment_name}"
        seg_used = helper_binary[seg_col].to_numpy(dtype=bool, copy=False)

        p10_values = np.zeros(len(helper_binary), dtype=float)
        median_values = np.zeros(len(helper_binary), dtype=float)
        p90_values = np.zeros(len(helper_binary), dtype=float)
        length_values = seg_used.astype(float) * float(segment_lengths_m[segment_name])

        # Only used segments can receive a missing historical value.
        p10_values[seg_used] = np.nan
        median_values[seg_used] = np.nan
        p90_values[seg_used] = np.nan

        segment_history = history_df.loc[
            history_df["segment_name"] == segment_name,
            ["taxi_end_ns", "n_points_in_segment"],
        ]

        if not segment_history.empty:
            segment_history = segment_history.sort_values("taxi_end_ns")

            historical_end_ns = segment_history["taxi_end_ns"].to_numpy(dtype=np.int64, copy=False)
            historical_durations = segment_history["n_points_in_segment"].to_numpy(dtype=float, copy=False)

            used_row_idx = np.flatnonzero(seg_used)

            for row_idx in used_row_idx:
                # side="left" enforces taxi_end < current taxi_start.
                cutoff = np.searchsorted(
                    historical_end_ns,
                    current_start_ns[row_idx],
                    side="left",
                )

                if cutoff == 0:
                    continue

                hist_values = historical_durations[:cutoff]

                p10, median, p90 = np.nanquantile(
                    hist_values,
                    [0.10, 0.50, 0.90],
                )

                p10_values[row_idx] = p10
                median_values[row_idx] = median
                p90_values[row_idx] = p90

        result_cols[f"p10_hist_seg_time_{segment_name}"] = p10_values
        result_cols[f"median_hist_seg_time_{segment_name}"] = median_values
        result_cols[f"p90_hist_seg_time_{segment_name}"] = p90_values
        result_cols[f"seg_length_m_{segment_name}"] = np.round(length_values, 1)

    return pd.DataFrame(result_cols)


# Compute leakage-free rolling historical segment statistics.
helper_bottleneck_features = compute_rolling_segment_quantiles(
    helper_binary=helper_bottleneck_features,
    valid_segment_hit_counts=valid_segment_hit_counts,
    segments_df=segments_df,
    segment_lengths_m=segment_lengths_m,
)

In [ ]:
helper_bottleneck_features.head()

In the following cell:
- Rolling historical taxiway segment times are summed (`sum_p10_hist_seg_time`, `sum_median_hist_seg_time`, and `sum_p90_hist_seg_time`) for each flight (`flight_id`). **If any used segment has one or more missing rolling historical times, the entire corresponding flight-level sum will be set to `NaN`**.
- Used taxiway segment lengths are summed for each flight (`sum_seg_length_m`).
- The number of intersections (= number of passed segments) is calculated for each flight (`n_intersections_passed`).
- `taxi_start` and `taxi_end` are kept for later leakage-free time-based splitting.

In [ ]:
p10_cols = [c for c in helper_bottleneck_features.columns if c.startswith("p10_hist_seg_time_")]
median_cols = [c for c in helper_bottleneck_features.columns if c.startswith("median_hist_seg_time_")]
p90_cols = [c for c in helper_bottleneck_features.columns if c.startswith("p90_hist_seg_time_")]
length_cols = [c for c in helper_bottleneck_features.columns if c.startswith("seg_length_m_")]

# If any actually used segment has NaN historical statistics, the corresponding sum becomes NaN.
helper_bottleneck_features["sum_p10_hist_seg_time"] = np.sum(
    helper_bottleneck_features[p10_cols].to_numpy(dtype=float, copy=False),
    axis=1,
)

helper_bottleneck_features["sum_median_hist_seg_time"] = np.sum(
    helper_bottleneck_features[median_cols].to_numpy(dtype=float, copy=False),
    axis=1,
)

helper_bottleneck_features["sum_p90_hist_seg_time"] = np.sum(
    helper_bottleneck_features[p90_cols].to_numpy(dtype=float, copy=False),
    axis=1,
)

helper_bottleneck_features["sum_seg_length_m"] = np.sum(
    helper_bottleneck_features[length_cols].to_numpy(dtype=float, copy=False),
    axis=1,
)

# Round the total segment length to 1 decimal place.
helper_bottleneck_features["sum_seg_length_m"] = (
    helper_bottleneck_features["sum_seg_length_m"].round(1)
)

# The number of passed intersections is defined here as the number of used segments.
helper_bottleneck_features["n_intersections_passed"] = (
    helper_bottleneck_features[length_cols]
    .gt(0)
    .sum(axis=1)
    .astype(np.uint16)
)

# Keep only the final aggregated features.
# taxi_start and taxi_end are kept for later leakage-free rolling train/test splits.
helper_bottleneck_features = helper_bottleneck_features.loc[
    :,
    [
        "flight_id",
        "taxi_start",
        "taxi_end",
        "sum_p10_hist_seg_time",
        "sum_median_hist_seg_time",
        "sum_p90_hist_seg_time",
        "sum_seg_length_m",
        "n_intersections_passed",
    ]
]

In [ ]:
helper_bottleneck_features.head()

In the following cell:
- The take-off runway value (`cut_runway`) is extracted for each flight (`flight_id`).
- The take-off runway value is one-hot encoded.
- The runway features are merged into the final feature DataFrame (`helper_bottleneck_features`).

In [ ]:
# Extract one runway value per flight.
runway_per_flight = (
    processed_df.loc[:, ["flight_id", "cut_runway"]]
    .dropna(subset=["flight_id"])
    .drop_duplicates(subset=["flight_id"])
    .rename(columns={"cut_runway": "TO_runway"})
)

# Create one-hot-encoded runway columns.
runway_per_flight["TO_runway"] = runway_per_flight["TO_runway"].astype(str)

runway_dummies = pd.get_dummies(
    runway_per_flight["TO_runway"],
    prefix="TO_runway",
    dtype=np.uint8,
)

# Keep only the required runway columns and ensure all exist.
required_runway_cols = [
    "TO_runway_28",
    "TO_runway_32",
    "TO_runway_16",
    "TO_runway_10",
    "TO_runway_34",
]

for col in required_runway_cols:
    if col not in runway_dummies.columns:
        runway_dummies[col] = 0

runway_dummies = runway_dummies.loc[:, required_runway_cols]

runway_per_flight = pd.concat(
    [
        runway_per_flight.loc[:, ["flight_id"]],
        runway_dummies,
    ],
    axis=1,
)

helper_bottleneck_features = helper_bottleneck_features.merge(
    runway_per_flight,
    on="flight_id",
    how="left",
)

# Fill missing runway values with 0.
helper_bottleneck_features[required_runway_cols] = (
    helper_bottleneck_features[required_runway_cols]
    .fillna(0)
    .astype(np.uint8)
)

# Reorder the columns.
helper_bottleneck_features = helper_bottleneck_features.loc[
    :,
    [
        "flight_id",
        "taxi_start",
        "taxi_end",
        "sum_p10_hist_seg_time",
        "sum_median_hist_seg_time",
        "sum_p90_hist_seg_time",
        "sum_seg_length_m",
        "n_intersections_passed",
        "TO_runway_28",
        "TO_runway_32",
        "TO_runway_16",
        "TO_runway_10",
        "TO_runway_34",
    ]
]

In [ ]:
helper_bottleneck_features.head(20)

In the following cell, the final feature DataFrame `helper_bottleneck_features` is saved as a parquet file (`helper_bottleneck_features.parquet`).

In [ ]:
output_path = "data/processed/helper_bottleneck_features.parquet"

# Save dataframe to parquet.
helper_bottleneck_features.to_parquet(output_path, index=False)

print(f"File \"helper_bottleneck_features\" saved to: {output_path}")